# รายงานผลการดำเนินงาน Sprint 2 (Final Term Project)

## ระบบค้นพบภาพยนตร์และสร้างรายการรับชม (Movie Discovery & Watchlist Builder)

**รายวิชา:** CP352301 Script Programming (1/2569)
**Sprint 2:** Back-End App Dev (สัปดาห์ที่ 13)
**กำหนดนำเสนอ:** 22–23/9/69
**กำหนดส่งงาน:** 25/9/69

| บทบาท (หมุนเวียนตาม Sprint) | สมาชิกในทีม |
|---|---|
| Planner / Team Leader | นางสาวพลอยชมพู วงศ์กีรติกุล |
| Coder | นายอเสข ปัญญาวงค์ |
| Debugger / QA | นายนวมินทร์ คำจันทร์ |

**Repository:** https://github.com/somjung/Movie-Discovery

---

เนื้อหาของรายงานแบ่งออกเป็น 4 ส่วน ดังนี้

1. **ส่วนที่ 1: บทสรุปโครงการ (Project Recap)** ได้แก่ ภาพรวมของระบบ สถาปัตยกรรม 3 ชั้น และสิ่งที่ Sprint 2 เพิ่มจากรอบก่อน
2. **ส่วนที่ 2: แผนงาน (Plan)** ได้แก่ เป้าหมาย ขอบเขต คำสั่งของระบบ และ Definition of Done ของ Sprint 2
3. **ส่วนที่ 3: การพัฒนา (Execution)** ได้แก่ โค้ดที่ส่งมอบทั้งชุด และสาธิตการทำงานจริงแบบออฟไลน์
4. **ส่วนที่ 4: ผลลัพธ์และการทดสอบ (Result)** ได้แก่ สรุปความก้าวหน้า ตาราง QA บทเรียน และลิงก์ของโปรเจกต์


## ส่วนที่ 1: บทสรุปโครงการ (Project Recap)

### 1. ภาพรวมระบบ

ระบบค้นพบภาพยนตร์และสร้างรายการรับชม (Movie Discovery & Watchlist Builder)
เป็นแอปพลิเคชันที่เปลี่ยน "ภาพยนตร์ที่ผู้ใช้ชื่นชอบ" ให้กลายเป็น "รายการรับชม" ที่รวบรวมไว้ในที่เดียว
ผู้ใช้ระบุภาพยนตร์ตั้งต้น (Seed Movie) หรือคำค้น ระบบจะดึงข้อมูลจริงจาก TMDB API (The Movie Database)
กรองและจัดอันดับตามเงื่อนไขที่ต้องการ บันทึกผลลงฐานข้อมูล SQLite ส่วนตัว แล้วส่งออกเป็นไฟล์ CSV

จุดเด่นของโครงงาน: ข้อมูลภาพยนตร์ที่คล้ายกันได้มาจาก API โดยตรง
ขณะที่ตรรกะการคัดเลือก การจัดอันดับ การจัดการคลังข้อมูล และการส่งออก เป็นส่วนที่ทีมพัฒนาขึ้นเองทั้งหมด

### 2. สถาปัตยกรรม 3 ชั้น (Separation of Concerns)

Sprint 1 ส่งมอบชั้นนำเสนอ (Presentation Layer) และวางโครงโมดูลทั้งหมดไว้ —
Sprint 2 เปิดใช้งานทั้งสามชั้นครบถ้วน โดยโค้ดแยกหน้าที่ชัดเจนและทิศทางการพึ่งพาชี้ลงล่างเสมอ:

```
        ┌──────────────────────────────────────────────┐
        │  Presentation Layer                          │
        │  cli.py · app.py                             │
        └───────────────────────┬──────────────────────┘
                                │  composes services
        ┌───────────────────────▼──────────────────────┐
        │  Business Logic Layer                        │
        │  discovery.py · watchlist.py                 │
        └───────────────────────┬──────────────────────┘
                                │  constructor injection
        ┌───────────────────────▼──────────────────────┐
        │  Data Access Layer                           │
        │  tmdb_client.py · movie_store.py · exporter.py │
        └──────────────────────────────────────────────┘
              │                  │               │
        TMDB REST API     data/movies.db    data/*.csv
```
*(ภาพที่ 1: โครงสร้าง 3 ชั้นและทิศทางการพึ่งพาของระบบหลัง Sprint 2)*

- **ชั้นนำเสนอ** — CLI รับคำสั่ง ตรวจสอบอินพุต แล้วแสดงผลเป็นข้อความไทย ไม่มีตรรกะการคัดเลือกอยู่ในชั้นนี้
- **ชั้นตรรกะธุรกิจ** — `DiscoveryService` (ค้นหา · กรอง · เรียง · ตัดรายการซ้ำ) และ `WatchlistBuilder` (เพิ่มรายการ · ดึงข้อมูลอัตโนมัติ) ทั้งสองคลาสรับ `TmdbClient` และ `MovieStore` ผ่านพารามิเตอร์ (constructor injection) จึงสลับของจริงกับของจำลองได้อย่างอิสระ
- **ชั้นเข้าถึงข้อมูล** — `TmdbClient` (TMDB + แคช 24 ชั่วโมง) · `MovieStore` (SQLite 6 ตาราง) · `Exporter` (ส่งออก CSV)

### 3. สิ่งที่ Sprint 2 เพิ่มจากรอบก่อน

- เชื่อมต่อ TMDB API จริงผ่านไลบรารี requests พร้อมแคชคำตอบ 24 ชั่วโมงในตาราง `api_cache` — ค้นคำเดิมซ้ำจะไม่ยิงเครือข่ายซ้ำ
- ฐานข้อมูล SQLite 6 ตาราง: `movies` · `similar_links` · `watchlist` · `favorites` · `searches` · `api_cache` — ข้อมูลคงอยู่ข้ามเซสชัน
- คำสั่งชุดเต็ม: `search` · `discover` พร้อมตัวเลือกกรอง/เรียง 5 แบบ · `watchlist add|list|remove|clear` · `favorites` · `export` CSV · `history`
- ข้อความแจ้งข้อผิดพลาดภาษาไทยครบทุกกรณี: ไม่มีคีย์ · คีย์ผิด (401) · ถูกจำกัดอัตรา (429) · ไม่พบข้อมูล (404) · เน็ตหลุด · ข้อมูลตอบกลับผิดรูป
- ชุดทดสอบออฟไลน์ 93 เคส (จาก 40 เคสใน Sprint 1) รันได้โดยไม่ใช้เน็ตและไม่ใช้คีย์
- ขนาดโค้ดชั้น `src/` ขยายจาก 23,803 เป็น 47,343 ไบต์ — พฤติกรรม CLI เดิมจาก Sprint 1 คงอยู่ทั้งหมด


## ส่วนที่ 2: แผนงาน (Plan)

### 2.1 เป้าหมายของ Sprint 2

พัฒนาชั้น Business Logic และ Data Access ให้ทำงานกับข้อมูลจริงจาก TMDB API และจัดเก็บข้อมูลอย่างยั่งยืนในฐานข้อมูล SQLite ประกอบด้วย 5 งานหลัก

1. เชื่อมต่อ TMDB API จริง — จัดการข้อผิดพลาดครบทุกกรณี และแคชผลลัพธ์ไว้ในตาราง `api_cache`
2. ฐานข้อมูล SQLite 6 ตาราง พร้อมใช้งานครบทุกฟังก์ชัน
3. ฟังก์ชันค้นหา (Searching) กรองข้อมูล (Filtering) และเรียงลำดับ (Sorting) ทำงานกับข้อมูลจริง
4. การจัดการ File I/O — ส่งออกรายการรับชมและรายการโปรดเป็นไฟล์ CSV
5. ชุดทดสอบอัตโนมัติแบบ Mock API — ต่อยอด CI เดิมให้เขียวต่อเนื่อง

### 2.2 ขอบเขตและข้อกำหนดคงเดิม

- **ในขอบเขต:** TMDB จริง (ค้นหา · หนังคล้าย · คำแนะนำ) พร้อมแคช · SQLite 6 ตาราง · คำสั่งกรอง/เรียง · ลิสต์เก็บถาวร · CSV · เทสต์แบบ Mock API
- **นอกขอบเขต (ยกไป Sprint 3):** การเชื่อมชั้นนำเสนอรูปแบบเต็มรูปแบบ และการรับมือกรณีขอบเขตเชิงลึกทั่วทั้งระบบ
- **ข้อกำหนดคงเดิม:** พฤติกรรมทั้งหมดจาก Sprint 1 ต้องอยู่ครบ (ข้อความไทย · การตรวจสอบอินพุต · รูปแบบการแสดงผล) และคีย์ API เก็บผ่านตัวแปรสภาพแวดล้อม `TMDB_API_KEY` เท่านั้น — ห้ามปรากฏในโค้ด เอกสาร หรือ commit

### 2.3 คำสั่งของระบบ (ฉบับเต็ม)

| คำสั่ง | พฤติกรรมที่คาดหวัง |
|---|---|
| `search <ชื่อเรื่อง>` | ค้นหาจริงจาก TMDB ผ่านแคช · บันทึกประวัติ · ไม่พบ → แจ้งให้ทราบ |
| `discover <รหัส> [ตัวเลือก]` | รวมหนังคล้าย + คำแนะนำ ตัดซ้ำ กรอง/เรียง/จำกัดจำนวน แล้วบันทึกลงฐานข้อมูล |
| ↳ `--min-rating` · `--min-votes` · `--year` | กรองตามคะแนนขั้นต่ำ · จำนวนโหวตขั้นต่ำ · ช่วงปีที่ฉาย |
| ↳ `--sort` · `--limit` | เรียง rating / votes / year / popularity · จำกัดจำนวน (ค่าเริ่มต้น 10) |
| `watchlist` (add · list · remove · clear) | จัดการรายการรับชม — ไม่เพิ่มซ้ำ · รหัสที่ยังไม่มีในเครื่องดึงจาก TMDB อัตโนมัติ |
| `favorites` (add · list · remove) | จัดการรายการโปรดในฐานข้อมูล |
| `export watchlist` / `export favorites` | เขียนไฟล์ CSV ลงโฟลเดอร์ `data/` แล้วแจ้งที่อยู่ไฟล์ |
| `history [N]` | ประวัติการค้นหาล่าสุด N รายการ (ค่าเริ่มต้น 10) |
| `help` · `quit` / `exit` / `q` / `ออก` | แสดงคำสั่งทั้งหมด · ออกจากโปรแกรมทันที |

### 2.4 Definition of Done (สรุป — ผลการตรวจอยู่ในส่วนที่ 4)

**การทำงานปกติ**

- search กับ TMDB จริง ผลถูกต้องและถูกบันทึกใน `searches` · ค้นซ้ำใช้แคชโดยไม่เรียก API ซ้ำ
- discover ทุกตัวเลือกกรอง/เรียง ผลตรงเงื่อนไข · watchlist และ favorites ทำงานครบ ไม่เพิ่มซ้ำ
- ปิด–เปิดโปรแกรมใหม่ข้อมูลอยู่ครบ · export ได้ไฟล์ CSV ที่เปิดอ่านได้ · history แสดงถูกต้อง

**กรณีผิดพลาดและขอบเขต**

- ไม่มีคีย์ · เน็ตหลุด · 401 · 429 · 404 — แจ้งข้อความอ่านง่าย โปรแกรมไม่หยุดทำงาน
- รหัสผิดรูปแบบ · คำสั่งว่าง · คำสั่งไม่รู้จัก — พฤติกรรมเหมือน Sprint 1 ทุกประการ
- ไฟล์ฐานข้อมูลถูกลบ — โปรแกรมสร้างใหม่ให้อัตโนมัติ ทำงานต่อได้ทันที

**คุณภาพโค้ด**

- flake8 ผ่าน · pytest ผ่านครบทั้งชุดโดยไม่ต้องใช้อินเทอร์เน็ต · CI เขียวต่อเนื่อง


## ส่วนที่ 3: การพัฒนา (Execution)

ลำดับการพัฒนา: ① ออกแบบสคีมา 6 ตารางและสัญญาบริการ → ② เขียนโมดูลชั้นข้อมูลและชั้นตรรกะธุรกิจใหม่ →
③ ขยาย CLI เป็นชุดคำสั่งเต็ม → ④ เขียนชุดทดสอบออฟไลน์แบบฉีดตัวจำลอง → ⑤ ตรวจ flake8 และเตรียม CI

โค้ดที่ส่งมอบทั้งชุดถูกฝังในเซลล์ด้านล่าง **ตามไฟล์จริงในโครงการทุกไบต์** —
โน้ตบุ๊กนี้สร้างด้วยสคริปต์ที่อ่านไฟล์จากโฟลเดอร์โค้ดโดยตรง รายงานกับโค้ดจึงไม่คลาดกัน
จากนั้นเซลล์สาธิตทั้งหมดจะรันจริงแบบ **ออฟไลน์ 100%** ด้วยตัวจำลอง TMDB ชุดเดียวกับชุดทดสอบ
(ไม่ต้องใช้คีย์หรืออินเทอร์เน็ต — รันซ้ำได้ทุกเครื่องและบน CI)


In [1]:
# เตรียมโครงสร้างโปรเจกต์และย้ายเข้าไปทำงานในโฟลเดอร์
import os
import pathlib
import sys

here = pathlib.Path.cwd()
if (here / "src" / "cli.py").exists():
    ROOT = here  # อยู่ในโฟลเดอร์โปรเจกต์อยู่แล้ว
else:
    ROOT = here / "script-final-project"
    if (ROOT / "src" / "cli.py").exists():
        # รันข้างโฟลเดอร์โปรเจกต์เดิม: ใช้โฟลเดอร์ใหม่ ไม่เขียนทับของเดิม
        ROOT = here / "script-final-project-notebook"
for folder in ("src", "tests", "data"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
ROOT = pathlib.Path.cwd()
for path in (ROOT, ROOT / "tests"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print("working directory:", ROOT)
print("python:", sys.version.split()[0])


working directory: C:\Users\User\AppData\Local\hermes\profiles\generalist\cache\scratch\nb_run\s2\script-final-project
python: 3.12.13


In [2]:
# ติดตั้งไลบรารีที่ใช้รันและทดสอบ (ถ้ามีอยู่แล้วระบบจะข้ามให้)
%pip install -q pytest flake8 requests
import pytest
print("pytest", pytest.__version__)


Note: you may need to restart the kernel to use updated packages.


pytest 9.1.1


In [3]:
%%writefile requirements.txt
requests>=2.31
pytest>=8.0
flake8>=7.0


Writing requirements.txt


In [4]:
%%writefile setup.cfg
[flake8]
max-line-length = 99
exclude = .venv,__pycache__,.git,.pytest_cache

[tool:pytest]
testpaths = tests


Writing setup.cfg


In [5]:
%%writefile .env.example
# TMDB API key (free for non-commercial use, attribution required).
# Create one at https://developer.themoviedb.org/ and paste it below,
# then copy this file to .env (the .env file is never committed).
TMDB_API_KEY=your_tmdb_api_key_here


Writing .env.example


In [6]:
%%writefile src/__init__.py
"""Movie Discovery & Watchlist Builder — core package."""

__version__ = "0.1.0"


Writing src/__init__.py


In [7]:
%%writefile src/config.py
"""Configuration constants and API-key lookup for the project."""

import os

TMDB_BASE_URL = "https://api.themoviedb.org/3"
DEFAULT_LANGUAGE = "en-US"
DEFAULT_TIMEOUT = 10

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def get_data_dir():
    """Folder for runtime data: the SQLite database and the CSV exports."""
    return os.path.join(PROJECT_ROOT, "data")


def get_db_path():
    """Path of the project's SQLite file (``data/movies.db`` under the root)."""
    return os.path.join(get_data_dir(), "movies.db")


def get_api_key():
    """Return the TMDB API key from the environment.

    Raises RuntimeError with a helpful message when the key is missing.
    """
    api_key = os.environ.get("TMDB_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError(
            "TMDB_API_KEY is not set. Create a free key at "
            "https://developer.themoviedb.org/ and export it, e.g. "
            "set TMDB_API_KEY=your_key_here"
        )
    return api_key


Writing src/config.py


In [8]:
%%writefile src/sample_data.py
"""Built-in sample dataset used by the Sprint 1 CLI and the test-suite.

The TMDB-backed data layer arrives in Sprint 2; until then the CLI is
demonstrated safely offline with these five movies.
"""

SAMPLE_MOVIES = [
    {"id": 101, "title": "Alpha Signal", "release_date": "2010-05-01",
     "vote_average": 7.8, "vote_count": 1200, "popularity": 40.0},
    {"id": 102, "title": "Bravo Horizon", "release_date": "2014-09-12",
     "vote_average": 6.1, "vote_count": 300, "popularity": 22.0},
    {"id": 103, "title": "Cobalt Night", "release_date": "1999-01-20",
     "vote_average": 8.4, "vote_count": 2100, "popularity": 55.0},
    {"id": 104, "title": "Delta Echo", "release_date": "",
     "vote_average": 5.0, "vote_count": 50, "popularity": 5.0},
    {"id": 105, "title": "Emerald Run", "release_date": "2021-11-05",
     "vote_average": 8.0, "vote_count": 900, "popularity": 30.0},
]


Writing src/sample_data.py


In [9]:
%%writefile src/tmdb_client.py
"""TMDB API client: read-only GET access with explicit error handling and an
optional SQLite response cache.
"""

import json
from urllib.parse import urlencode

import requests

from .config import TMDB_BASE_URL, DEFAULT_LANGUAGE, DEFAULT_TIMEOUT, get_api_key
from .movie_store import CACHE_TTL_SECONDS


class TmdbError(Exception):
    """Raised when a TMDB request cannot be completed."""


class TmdbClient:
    """Minimal TMDB REST client (only the endpoints this project needs).

    Pass a ``store`` to enable response caching in its ``api_cache`` table:
    a request that is still fresh is served from SQLite without any network
    call; everything else is fetched and then stored for next time.
    """

    def __init__(self, api_key=None, base_url=TMDB_BASE_URL,
                 timeout=DEFAULT_TIMEOUT, session=None, store=None,
                 cache_ttl=CACHE_TTL_SECONDS):
        """Build a client; ``store=None`` disables the response cache."""
        self.api_key = api_key if api_key is not None else get_api_key()
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.session = session if session is not None else requests.Session()
        self.store = store
        self.cache_ttl = cache_ttl
        # Counters the CLI and tests can read to show the cache working.
        self.cache_hits = 0
        self.cache_misses = 0

    def _cache_key(self, path, query):
        """Deterministic key: the path plus every parameter except the API key."""
        public = sorted((key, value) for key, value in query.items()
                        if key != "api_key")
        return f"{path}?{urlencode(public)}"

    def _read_cache(self, cache_key):
        """Fresh cached body as a dict, or None (absent, expired, or corrupt)."""
        if self.store is None:
            return None
        cached = self.store.get_cached(cache_key, self.cache_ttl)
        if cached is None:
            return None
        try:
            return json.loads(cached)
        except ValueError:
            return None  # corrupt entry: ignore it and refetch from the API

    def _write_cache(self, cache_key, payload):
        """Store a fresh response body for future reads (no-op without a store)."""
        if self.store is not None:
            self.store.save_cached(cache_key, json.dumps(payload))

    def _get(self, path, params=None):
        """GET ``path`` and return the decoded JSON body.

        Served from the cache when fresh; otherwise fetched from the network
        and stored. Every failure mode raises TmdbError with a readable
        message: network problems, auth failures, missing resources, rate
        limits, unexpected statuses, and invalid JSON.
        """
        query = {"api_key": self.api_key, "language": DEFAULT_LANGUAGE}
        if params:
            query.update(params)
        cache_key = self._cache_key(path, query)

        cached = self._read_cache(cache_key)
        if cached is not None:
            self.cache_hits += 1
            return cached
        if self.store is not None:
            self.cache_misses += 1

        try:
            response = self.session.get(
                f"{self.base_url}{path}", params=query, timeout=self.timeout
            )
        except requests.RequestException as exc:
            raise TmdbError(f"request to {path} failed: {exc}") from exc

        if response.status_code == 401:
            raise TmdbError("TMDB rejected the API key (HTTP 401)")
        if response.status_code == 404:
            raise TmdbError(f"TMDB resource not found (HTTP 404): {path}")
        if response.status_code == 429:
            raise TmdbError("TMDB rate limit hit (HTTP 429) - slow down")
        if response.status_code != 200:
            raise TmdbError(
                f"unexpected TMDB status {response.status_code} for {path}"
            )
        try:
            payload = response.json()
        except ValueError as exc:
            raise TmdbError(f"TMDB returned invalid JSON for {path}") from exc
        self._write_cache(cache_key, payload)
        return payload

    def search_movies(self, query, page=1):
        """Search movies by title. Returns a list of movie dicts."""
        data = self._get("/search/movie", {"query": query, "page": page})
        return data.get("results", [])

    def get_movie_details(self, tmdb_id):
        """Full details for one movie id (used when adding it by id)."""
        return self._get(f"/movie/{tmdb_id}")

    def get_similar(self, tmdb_id, page=1):
        """Movies similar to a given movie id."""
        data = self._get(f"/movie/{tmdb_id}/similar", {"page": page})
        return data.get("results", [])

    def get_recommendations(self, tmdb_id, page=1):
        """TMDB's recommendations for a given movie id."""
        data = self._get(f"/movie/{tmdb_id}/recommendations", {"page": page})
        return data.get("results", [])


Writing src/tmdb_client.py


In [10]:
%%writefile src/movie_store.py
"""SQLite persistence layer: movies, similar-links, watchlist, favorites,
search history, and the TMDB response cache.
"""

import os
import sqlite3

# How long a cached TMDB response stays fresh (24 hours, in seconds).
CACHE_TTL_SECONDS = 24 * 60 * 60

SCHEMA = """
CREATE TABLE IF NOT EXISTS movies (
    tmdb_id      INTEGER PRIMARY KEY,
    title        TEXT NOT NULL,
    release_date TEXT,
    vote_average REAL,
    vote_count   INTEGER,
    popularity   REAL,
    overview     TEXT,
    added_at     TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS similar_links (
    source_id INTEGER NOT NULL,
    target_id INTEGER NOT NULL,
    score     REAL,
    PRIMARY KEY (source_id, target_id)
);

CREATE TABLE IF NOT EXISTS watchlist (
    movie_id INTEGER PRIMARY KEY,
    note     TEXT,
    added_at TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS favorites (
    movie_id INTEGER PRIMARY KEY,
    note     TEXT,
    added_at TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS searches (
    id           INTEGER PRIMARY KEY AUTOINCREMENT,
    query        TEXT NOT NULL,
    result_count INTEGER NOT NULL DEFAULT 0,
    searched_at  TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS api_cache (
    cache_key  TEXT PRIMARY KEY,
    payload    TEXT NOT NULL,
    fetched_at TEXT NOT NULL DEFAULT (datetime('now'))
);
"""


class MovieStore:
    """Small data-access layer on top of SQLite."""

    def __init__(self, db_path=":memory:"):
        """Open a connection and create the tables when they do not exist yet."""
        if db_path != ":memory:":
            parent = os.path.dirname(os.path.abspath(db_path))
            os.makedirs(parent, exist_ok=True)
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row
        self.init_db()

    def close(self):
        """Close the underlying database connection."""
        self.conn.close()

    def init_db(self):
        """Create the tables when they do not exist yet."""
        with self.conn:
            self.conn.executescript(SCHEMA)

    def add_movie(self, movie):
        """Insert a movie (or refresh it when the id already exists)."""
        with self.conn:
            self.conn.execute(
                """
                INSERT INTO movies (tmdb_id, title, release_date, vote_average,
                                    vote_count, popularity, overview)
                VALUES (:tmdb_id, :title, :release_date, :vote_average,
                        :vote_count, :popularity, :overview)
                ON CONFLICT(tmdb_id) DO UPDATE SET
                    title = excluded.title,
                    release_date = excluded.release_date,
                    vote_average = excluded.vote_average,
                    vote_count = excluded.vote_count,
                    popularity = excluded.popularity,
                    overview = excluded.overview
                """,
                {
                    "tmdb_id": movie["id"],
                    "title": movie.get("title", ""),
                    "release_date": movie.get("release_date") or "",
                    "vote_average": movie.get("vote_average") or 0.0,
                    "vote_count": movie.get("vote_count") or 0,
                    "popularity": movie.get("popularity") or 0.0,
                    "overview": movie.get("overview") or "",
                },
            )
        return movie["id"]

    def get_movie(self, tmdb_id):
        """Return one movie as a dict, or None when it is not stored."""
        row = self.conn.execute(
            "SELECT * FROM movies WHERE tmdb_id = ?", (tmdb_id,)
        ).fetchone()
        return dict(row) if row else None

    def list_movies(self):
        """All stored movies (most recently added first)."""
        rows = self.conn.execute(
            "SELECT * FROM movies ORDER BY added_at DESC, tmdb_id"
        ).fetchall()
        return [dict(row) for row in rows]

    def add_similar_link(self, source_id, target_id, score=None):
        """Record that ``target_id`` is similar to ``source_id``."""
        with self.conn:
            self.conn.execute(
                """
                INSERT INTO similar_links (source_id, target_id, score)
                VALUES (?, ?, ?)
                ON CONFLICT(source_id, target_id) DO UPDATE SET
                    score = excluded.score
                """,
                (source_id, target_id, score),
            )

    def get_similar_links(self, source_id):
        """Similar-movie links recorded for a source movie (best first)."""
        rows = self.conn.execute(
            """
            SELECT target_id, score FROM similar_links
            WHERE source_id = ? ORDER BY score DESC
            """,
            (source_id,),
        ).fetchall()
        return [dict(row) for row in rows]

    def add_to_watchlist(self, movie_id, note=None):
        """Add a movie to the watchlist (no duplicates)."""
        with self.conn:
            self.conn.execute(
                """
                INSERT INTO watchlist (movie_id, note) VALUES (?, ?)
                ON CONFLICT(movie_id) DO UPDATE SET note = excluded.note
                """,
                (movie_id, note),
            )

    def is_in_watchlist(self, movie_id):
        """True when the movie id is already on the watchlist."""
        row = self.conn.execute(
            "SELECT 1 FROM watchlist WHERE movie_id = ?", (movie_id,)
        ).fetchone()
        return row is not None

    def get_watchlist(self):
        """Watchlist rows joined with movie details (newest first)."""
        rows = self.conn.execute(
            """
            SELECT w.movie_id AS tmdb_id, m.title, m.release_date,
                   m.vote_average, w.note, w.added_at
            FROM watchlist w
            LEFT JOIN movies m ON m.tmdb_id = w.movie_id
            ORDER BY w.added_at DESC, w.movie_id
            """
        ).fetchall()
        return [dict(row) for row in rows]

    def remove_from_watchlist(self, movie_id):
        """Delete one watchlist row; return True when a row was removed."""
        with self.conn:
            cur = self.conn.execute(
                "DELETE FROM watchlist WHERE movie_id = ?", (movie_id,)
            )
        return cur.rowcount > 0

    def clear_watchlist(self):
        """Delete every watchlist row; return how many entries were removed."""
        with self.conn:
            cur = self.conn.execute("DELETE FROM watchlist")
        return cur.rowcount

    def add_favorite(self, movie_id, note=None):
        """Add a movie to favorites (no duplicates; refreshes the note)."""
        with self.conn:
            self.conn.execute(
                """
                INSERT INTO favorites (movie_id, note) VALUES (?, ?)
                ON CONFLICT(movie_id) DO UPDATE SET note = excluded.note
                """,
                (movie_id, note),
            )

    def is_in_favorites(self, movie_id):
        """True when the movie id is already in favorites."""
        row = self.conn.execute(
            "SELECT 1 FROM favorites WHERE movie_id = ?", (movie_id,)
        ).fetchone()
        return row is not None

    def get_favorites(self):
        """Favorite rows joined with movie details (newest first)."""
        rows = self.conn.execute(
            """
            SELECT f.movie_id AS tmdb_id, m.title, m.release_date,
                   m.vote_average, f.note, f.added_at
            FROM favorites f
            LEFT JOIN movies m ON m.tmdb_id = f.movie_id
            ORDER BY f.added_at DESC, f.movie_id
            """
        ).fetchall()
        return [dict(row) for row in rows]

    def remove_favorite(self, movie_id):
        """Delete one favorites row; return True when a row was removed."""
        with self.conn:
            cur = self.conn.execute(
                "DELETE FROM favorites WHERE movie_id = ?", (movie_id,)
            )
        return cur.rowcount > 0

    def record_search(self, query, result_count):
        """Append one search to the history (query text + number of results)."""
        with self.conn:
            self.conn.execute(
                "INSERT INTO searches (query, result_count) VALUES (?, ?)",
                (query, result_count),
            )

    def get_recent_searches(self, limit=10):
        """Return the most recent searches first, up to ``limit`` rows."""
        rows = self.conn.execute(
            """
            SELECT id, query, result_count, searched_at FROM searches
            ORDER BY id DESC LIMIT ?
            """,
            (limit,),
        ).fetchall()
        return [dict(row) for row in rows]

    def get_cached(self, cache_key, ttl_seconds=CACHE_TTL_SECONDS):
        """Return a cached payload younger than the TTL, or None on a miss."""
        row = self.conn.execute(
            """
            SELECT payload FROM api_cache
            WHERE cache_key = ? AND fetched_at >= datetime('now', ?)
            """,
            (cache_key, f"-{max(0, int(ttl_seconds))} seconds"),
        ).fetchone()
        return row["payload"] if row else None

    def save_cached(self, cache_key, payload):
        """Store (or refresh) a response payload under ``cache_key``."""
        with self.conn:
            self.conn.execute(
                """
                INSERT INTO api_cache (cache_key, payload) VALUES (?, ?)
                ON CONFLICT(cache_key) DO UPDATE SET
                    payload = excluded.payload,
                    fetched_at = datetime('now')
                """,
                (cache_key, payload),
            )


Writing src/movie_store.py


In [11]:
%%writefile src/discovery.py
"""Discovery logic: search, filter, sort, dedupe and rank movie candidates."""

SORT_OPTIONS = ("rating", "votes", "year", "popularity")


class DiscoveryService:
    """Turns user queries and seed ids into ranked candidate lists.

    The API only supplies raw search/similar/recommended movies; which of
    them survive the filters and in which order is decided here (project
    logic), and everything shown to the user is kept in the local store.
    """

    def __init__(self, client, store):
        """Keep the API client and the local store the service writes to."""
        self.client = client
        self.store = store

    def search(self, query, limit=10):
        """Search TMDB for ``query``, log it in ``searches``, return top hits."""
        results = self.client.search_movies(query)
        self.store.record_search(query, len(results))
        return results[:limit]

    def find_similar(self, seed_id, min_rating=0.0, min_votes=0,
                     year_from=None, year_to=None, limit=10, sort="rating"):
        """Return the top ``limit`` candidates for the seed movie.

        Candidates come from the similar + recommended endpoints (merged,
        with duplicates removed), then are filtered, sorted by ``sort``
        and cut down to ``limit`` before being saved to the store.
        """
        candidates = {}
        for fetch in (self.client.get_similar, self.client.get_recommendations):
            for movie in fetch(seed_id):
                candidates.setdefault(movie["id"], movie)

        kept = [
            movie for movie in candidates.values()
            if self._passes_filters(movie, min_rating, min_votes,
                                    year_from, year_to)
        ]
        kept.sort(key=self._sort_key(sort), reverse=True)
        kept = kept[:limit]

        for movie in kept:
            self.store.add_movie(movie)
            self.store.add_similar_link(seed_id, movie["id"],
                                        movie.get("vote_average"))
        return kept

    @classmethod
    def _sort_key(cls, sort):
        """Pick the (primary, secondary) sort key; reject unknown names."""
        if sort not in SORT_OPTIONS:
            raise ValueError(f"unknown sort option: {sort!r}")
        keys = {
            "rating": lambda movie: (movie.get("vote_average") or 0,
                                     movie.get("vote_count") or 0),
            "votes": lambda movie: (movie.get("vote_count") or 0,
                                    movie.get("vote_average") or 0),
            "year": lambda movie: (cls.release_year(movie),
                                   movie.get("vote_average") or 0),
            "popularity": lambda movie: (movie.get("popularity") or 0,
                                         movie.get("vote_average") or 0),
        }
        return keys[sort]

    @staticmethod
    def release_year(movie):
        """Release year as int, or 0 when the date is unknown/invalid."""
        date = (movie.get("release_date") or "")[:4]
        return int(date) if date.isdigit() else 0

    @classmethod
    def _passes_filters(cls, movie, min_rating, min_votes, year_from, year_to):
        """True when the movie satisfies every active filter."""
        if (movie.get("vote_average") or 0) < min_rating:
            return False
        if (movie.get("vote_count") or 0) < min_votes:
            return False
        year = cls.release_year(movie)
        if year == 0 and (year_from or year_to):
            return False  # an unknown date cannot be placed inside a range
        if year_from and year < year_from:
            return False
        if year_to and year > year_to:
            return False
        return True


Writing src/discovery.py


In [12]:
%%writefile src/watchlist.py
"""Watchlist and favorites logic: Top-N building and single-movie adds."""


class WatchlistBuilder:
    """Builds the user's Top-N watchlist and adds movies by id.

    The optional ``client`` lets ``add``/``add_favorite`` fetch a movie
    from TMDB when it is not stored locally yet.
    """

    def __init__(self, store, client=None):
        """Keep the store and an optional TMDB client for fetch-by-id."""
        self.store = store
        self.client = client

    def build(self, candidates, top_n=10):
        """Pick up to ``top_n`` new movies and save them to the watchlist."""
        picked = []
        for movie in candidates:
            if self.store.is_in_watchlist(movie["id"]):
                continue
            picked.append(movie)
            if len(picked) >= top_n:
                break
        for movie in picked:
            self.store.add_to_watchlist(movie["id"])
        return picked

    def ensure_movie(self, movie_id):
        """Return the stored movie, fetching it from TMDB when it is missing."""
        movie = self.store.get_movie(movie_id)
        if movie is not None:
            return movie
        if self.client is None:
            raise ValueError(
                f"movie {movie_id} is not stored and no TMDB client is attached"
            )
        self.store.add_movie(self.client.get_movie_details(movie_id))
        return self.store.get_movie(movie_id)

    def add(self, movie_id, note=None):
        """Add one movie to the watchlist by id (fetching it when needed).

        Returns True when it was newly added, and False when it was
        already on the watchlist.
        """
        self.ensure_movie(movie_id)
        already = self.store.is_in_watchlist(movie_id)
        self.store.add_to_watchlist(movie_id, note)
        return not already

    def add_favorite(self, movie_id, note=None):
        """Add one movie to favorites by id (fetching it when needed).

        Returns True when it was newly added, and False when it was
        already in favorites.
        """
        self.ensure_movie(movie_id)
        already = self.store.is_in_favorites(movie_id)
        self.store.add_favorite(movie_id, note)
        return not already


Writing src/watchlist.py


In [13]:
%%writefile src/exporter.py
"""CSV export of the user's lists (watchlist and favorites)."""

import csv
import os

LIST_COLUMNS = ("tmdb_id", "title", "release_date", "vote_average", "note")
# Both lists share the same export schema; the old name stays for safety.
WATCHLIST_COLUMNS = LIST_COLUMNS


class ExportError(Exception):
    """Raised when a list cannot be written to disk."""


class Exporter:
    """Writes project data to CSV files."""

    @staticmethod
    def _write_rows(rows, path, columns):
        """Write ``rows`` to ``path``, creating the folder chain first.

        Raises ExportError (naming the failing path) when writing fails.
        """
        try:
            folder = os.path.dirname(os.path.abspath(path))
            os.makedirs(folder, exist_ok=True)
            with open(path, "w", newline="", encoding="utf-8") as handle:
                writer = csv.writer(handle)
                writer.writerow(columns)
                for row in rows:
                    writer.writerow([
                        "" if row.get(column) is None else row.get(column)
                        for column in columns
                    ])
        except OSError as exc:
            raise ExportError(f"cannot write {path}: {exc}") from exc

    @staticmethod
    def export_watchlist(store, path):
        """Write the current watchlist to ``path``; return how many rows."""
        rows = store.get_watchlist()
        Exporter._write_rows(rows, path, LIST_COLUMNS)
        return len(rows)

    @staticmethod
    def export_favorites(store, path):
        """Write the current favorites to ``path``; return how many rows."""
        rows = store.get_favorites()
        Exporter._write_rows(rows, path, LIST_COLUMNS)
        return len(rows)


Writing src/exporter.py


In [14]:
%%writefile src/cli.py
"""Interactive command-line front-end for the Movie Discovery app.

Sprint 2 scope: the presentation layer is now wired to the real back-end —
TMDB (with caching) for search/discover, SQLite for watchlist/favorites/
search history, and CSV export for the File I/O requirement.  Commands keep
the Sprint-1 Thai messages and validation; ``client`` and ``store`` can be
injected so the test-suite runs fully offline.
"""

import os

from .config import get_data_dir, get_db_path
from .discovery import SORT_OPTIONS, DiscoveryService
from .exporter import Exporter, ExportError
from .movie_store import MovieStore
from .tmdb_client import TmdbClient, TmdbError
from .watchlist import WatchlistBuilder

WELCOME_MESSAGE = """\
============================================================
  Movie Discovery & Watchlist Builder — CLI
  ระบบค้นพบภาพยนตร์และสร้างรายการรับชม
============================================================"""

MENU_TEXT = """\
คำสั่งที่ใช้ได้:
  search <ชื่อเรื่อง>              ค้นหาภาพยนตร์จาก TMDB
  discover <รหัส> [ตัวเลือก]       ภาพยนตร์ที่คล้ายกัน (กรอง/เรียงได้)
        --min-rating 0-10 · --min-votes N · --year YYYY|YYYY-YYYY
        --sort rating|votes|year|popularity · --limit N
  watchlist add <รหัส>            เพิ่มเข้ารายการรับชม
  watchlist list                  แสดงรายการรับชม
  watchlist remove <รหัส>         นำออกจากรายการรับชม
  watchlist clear                 ล้างรายการรับชมทั้งหมด
  favorites add <รหัส>            เพิ่มเข้ารายการโปรด
  favorites list                  แสดงรายการโปรด
  favorites remove <รหัส>         นำออกจากรายการโปรด
  export watchlist|favorites      ส่งออกรายการเป็น CSV (โฟลเดอร์ data/)
  history [จำนวน]                 ประวัติการค้นหาล่าสุด (ค่าเริ่มต้น 10)
  help                            แสดงคำสั่งทั้งหมด
  quit / exit / ออก               ออกจากโปรแกรม"""

FAREWELL_MESSAGE = "ขอบคุณที่ใช้งานโปรแกรม ลาก่อน"
QUIT_COMMANDS = {"quit", "exit", "q", "ออก"}

MISSING_KEY_MESSAGE = (
    "ยังไม่ได้ตั้งค่าคีย์ TMDB — กำหนดตัวแปรสภาพแวดล้อม TMDB_API_KEY "
    "ก่อนใช้งาน\n"
    "  · สร้างคีย์ฟรีได้ที่ https://developer.themoviedb.org/\n"
    "  · คำสั่งที่ไม่ใช้ TMDB (watchlist list, export, history, help) "
    "ยังใช้งานได้ตามปกติ"
)

DISCOVER_FLAGS = ("--min-rating", "--min-votes", "--year", "--sort", "--limit")

LIST_LABELS = {"watchlist": "รายการรับชม", "favorites": "รายการโปรด"}


class CLI:
    """Presentation layer of the project (menus + input validation)."""

    def __init__(self, input_func=input, output_func=print, client=None,
                 store=None, db_path=None, data_dir=None):
        """Wire the CLI; inject ``client``/``store`` to run offline (tests).

        Without a ``client`` the real TMDB client is built from
        ``TMDB_API_KEY``; when the key is missing the CLI still starts and
        only the TMDB-backed commands become unavailable.
        """
        self.input_func = input_func
        self.output_func = output_func
        self.store = (store if store is not None
                      else MovieStore(db_path or get_db_path()))
        self.data_dir = data_dir or get_data_dir()
        self.key_missing = False
        if client is not None:
            self.client = client
        else:
            try:
                self.client = TmdbClient(store=self.store)
            except RuntimeError:
                self.client = None
                self.key_missing = True
        self.discovery = (DiscoveryService(self.client, self.store)
                          if self.client is not None else None)
        self.builder = WatchlistBuilder(self.store, self.client)
        self.running = False

    # ---------- display ----------

    def display_welcome_message(self):
        """Show the welcome banner."""
        self.output_func(WELCOME_MESSAGE)

    def display_menu(self):
        """Show the list of available commands."""
        self.output_func(MENU_TEXT)

    # ---------- input ----------

    def get_command_input(self):
        """Read one command line and normalise it (strip + lowercase)."""
        raw = self.input_func("movie> ")
        return raw.strip().lower()

    # ---------- main loop ----------

    def run(self):
        """Main loop: read -> validate -> dispatch, without ever crashing."""
        self.display_welcome_message()
        self.display_menu()
        self.running = True
        while self.running:
            try:
                self.handle_command(self.get_command_input())
            except EOFError:
                self.output_func("")
                self.output_func("[สิ้นสุดอินพุต]")
                self.running = False
            except KeyboardInterrupt:
                self.output_func("")
                self.output_func("[ยกเลิกโดยผู้ใช้]")
                self.running = False
            except ValueError as exc:
                self.output_func(f"คำเตือน: {exc}")
            except TmdbError as exc:
                self.output_func(self.describe_tmdb_error(exc))
            except ExportError as exc:
                self.output_func(f"ไม่สามารถเขียนไฟล์ได้: {exc}")
        self.output_func(FAREWELL_MESSAGE)

    def handle_command(self, text):
        """Validate and dispatch one normalised command line."""
        if not text:
            raise ValueError("กรุณาพิมพ์คำสั่ง (พิมพ์ help เพื่อดูคำสั่งทั้งหมด)")
        parts = text.split()
        name, args = parts[0], parts[1:]
        if name in QUIT_COMMANDS:
            self.running = False
            return
        handlers = {
            "search": self.cmd_search,
            "discover": self.cmd_discover,
            "watchlist": self.cmd_watchlist,
            "favorites": self.cmd_favorites,
            "export": self.cmd_export,
            "history": self.cmd_history,
            "help": self.cmd_help,
        }
        handler = handlers.get(name)
        if handler is None:
            raise ValueError(
                f"ไม่รู้จักคำสั่ง '{name}' (พิมพ์ help เพื่อดูคำสั่งทั้งหมด)"
            )
        handler(args)

    # ---------- command handlers ----------

    def cmd_help(self, args):
        """help — show the command menu again."""
        self.display_menu()

    def cmd_search(self, args):
        """search <ชื่อเรื่อง> — search TMDB (cached) and log the query."""
        if not args:
            raise ValueError("คำสั่ง search ต้องระบุชื่อเรื่อง เช่น search alpha")
        if self.client is None:
            self.output_func(MISSING_KEY_MESSAGE)
            return
        query = " ".join(args)
        results = self.discovery.search(query)
        if not results:
            self.output_func(f"ไม่พบภาพยนตร์ที่ตรงกับ '{query}'")
            return
        self.output_func(f"พบ {len(results)} เรื่อง (แสดงสูงสุด 10 รายการแรก):")
        for movie in results:
            self.output_func("  " + self.format_movie(movie))

    def cmd_discover(self, args):
        """discover <รหัส> [ตัวเลือก] — similar movies, filtered and sorted."""
        if not args:
            raise ValueError(
                "คำสั่ง discover ต้องระบุรหัสภาพยนตร์ เช่น discover 101"
            )
        seed_id = self.parse_positive_int(args[0], "รหัสภาพยนตร์")
        options = self.parse_discover_options(args[1:])
        if self.client is None:
            self.output_func(MISSING_KEY_MESSAGE)
            return
        try:
            seed = self.builder.ensure_movie(seed_id)
        except TmdbError as exc:
            if "404" in str(exc):
                self.output_func(f"ไม่พบรหัสภาพยนตร์ {seed_id} ใน TMDB")
                return
            raise
        results = self.discovery.find_similar(
            seed_id,
            min_rating=options["min_rating"],
            min_votes=options["min_votes"],
            year_from=options["year_from"],
            year_to=options["year_to"],
            limit=options["limit"],
            sort=options["sort"],
        )
        if not results:
            self.output_func(
                f"ไม่พบภาพยนตร์ที่ผ่านเงื่อนไขสำหรับรหัส {seed_id}"
            )
            return
        self.output_func(
            f"ภาพยนตร์ที่คล้ายกับ '{seed['title']}' "
            f"({len(results)} เรื่อง · เรียงตาม {options['sort']}):"
        )
        for rank, movie in enumerate(results, start=1):
            self.output_func(f"  {rank}. {self.format_movie(movie)}")

    def cmd_watchlist(self, args):
        """watchlist add <รหัส> | list | remove <รหัส> | clear."""
        if not args:
            raise ValueError(
                "คำสั่ง watchlist ต้องมีคำสั่งย่อย: "
                "add <รหัส> | list | remove <รหัส> | clear"
            )
        action, rest = args[0], args[1:]
        if action == "list":
            self.show_list("watchlist")
        elif action == "add":
            self.add_to_list("watchlist", rest)
        elif action == "remove":
            self.remove_from_list("watchlist", rest)
        elif action == "clear":
            count = self.store.clear_watchlist()
            self.output_func(f"ล้างรายการรับชมเรียบร้อยแล้ว ({count} เรื่อง)")
        else:
            raise ValueError(
                f"ไม่รู้จักคำสั่งย่อย '{action}' "
                "(ใช้ add <รหัส> | list | remove <รหัส> | clear)"
            )

    def cmd_favorites(self, args):
        """favorites add <รหัส> | list | remove <รหัส>."""
        if not args:
            raise ValueError(
                "คำสั่ง favorites ต้องมีคำสั่งย่อย: "
                "add <รหัส> | list | remove <รหัส>"
            )
        action, rest = args[0], args[1:]
        if action == "list":
            self.show_list("favorites")
        elif action == "add":
            self.add_to_list("favorites", rest)
        elif action == "remove":
            self.remove_from_list("favorites", rest)
        else:
            raise ValueError(
                f"ไม่รู้จักคำสั่งย่อย '{action}' "
                "(ใช้ add <รหัส> | list | remove <รหัส>)"
            )

    def cmd_export(self, args):
        """export watchlist|favorites — write the list to CSV under data/."""
        if not args or args[0] not in LIST_LABELS:
            raise ValueError(
                "คำสั่ง export ต้องระบุรายการ: "
                "export watchlist หรือ export favorites"
            )
        kind = args[0]
        path = os.path.join(self.data_dir, f"{kind}.csv")
        if kind == "watchlist":
            count = Exporter.export_watchlist(self.store, path)
        else:
            count = Exporter.export_favorites(self.store, path)
        self.output_func(f"ส่งออก {count} รายการไปที่ {path}")

    def cmd_history(self, args):
        """history [N] — show the most recent searches (default 10)."""
        limit = 10
        if args:
            limit = self.parse_positive_int(args[0], "จำนวนรายการ")
        rows = self.store.get_recent_searches(limit)
        if not rows:
            self.output_func("ยังไม่มีประวัติการค้นหา")
            return
        self.output_func(f"ประวัติการค้นหาล่าสุด ({len(rows)} รายการ):")
        for index, row in enumerate(rows, start=1):
            self.output_func(
                f"  {index}. '{row['query']}' — พบ {row['result_count']} ผล "
                f"({row['searched_at']})"
            )

    # ---------- list helpers ----------

    def show_list(self, kind):
        """Print one stored list (``watchlist`` or ``favorites``)."""
        label = LIST_LABELS[kind]
        rows = (self.store.get_watchlist() if kind == "watchlist"
                else self.store.get_favorites())
        if not rows:
            self.output_func(f"{label}ยังว่างอยู่ (ใช้ {kind} add <รหัส>)")
            return
        self.output_func(f"{label} ({len(rows)} เรื่อง):")
        for row in rows:
            self.output_func("  " + self.format_movie(row))

    def add_to_list(self, kind, rest):
        """Add one movie id to the watchlist or favorites (fetch if needed)."""
        label = LIST_LABELS[kind]
        if not rest:
            raise ValueError(
                f"คำสั่ง {kind} add ต้องระบุรหัสภาพยนตร์ เช่น {kind} add 101"
            )
        movie_id = self.parse_positive_int(rest[0], "รหัสภาพยนตร์")
        if self.client is None and self.store.get_movie(movie_id) is None:
            self.output_func(MISSING_KEY_MESSAGE)
            return
        try:
            movie = self.builder.ensure_movie(movie_id)
        except TmdbError as exc:
            if "404" in str(exc):
                self.output_func(f"ไม่พบรหัสภาพยนตร์ {movie_id} ใน TMDB")
                return
            raise
        if kind == "watchlist":
            added = self.builder.add(movie_id)
        else:
            added = self.builder.add_favorite(movie_id)
        if added:
            self.output_func(f"เพิ่ม '{movie['title']}' เข้า{label}แล้ว")
        else:
            self.output_func(f"'{movie['title']}' อยู่ใน{label}แล้ว")

    def remove_from_list(self, kind, rest):
        """Remove one movie id from the watchlist or favorites."""
        label = LIST_LABELS[kind]
        if not rest:
            raise ValueError(
                f"คำสั่ง {kind} remove ต้องระบุรหัสภาพยนตร์ เช่น "
                f"{kind} remove 101"
            )
        movie_id = self.parse_positive_int(rest[0], "รหัสภาพยนตร์")
        if kind == "watchlist":
            removed = self.store.remove_from_watchlist(movie_id)
        else:
            removed = self.store.remove_favorite(movie_id)
        if removed:
            movie = self.store.get_movie(movie_id)
            title = movie["title"] if movie else str(movie_id)
            self.output_func(f"นำ '{title}' ออกจาก{label}แล้ว")
        else:
            self.output_func(f"ไม่พบรหัสภาพยนตร์ {movie_id} ใน{label}")

    # ---------- option / value parsing ----------

    def parse_discover_options(self, tokens):
        """Parse discover flags (--min-rating ... --limit) into a dict."""
        options = {"min_rating": 0.0, "min_votes": 0, "year_from": None,
                   "year_to": None, "sort": "rating", "limit": 10}
        index = 0
        while index < len(tokens):
            flag = tokens[index]
            if flag not in DISCOVER_FLAGS:
                raise ValueError(
                    f"ไม่รู้จักตัวเลือก '{flag}' "
                    f"(ใช้ {' · '.join(DISCOVER_FLAGS)})"
                )
            if index + 1 >= len(tokens):
                raise ValueError(f"ตัวเลือก {flag} ต้องมีค่าตามหลัง")
            value = tokens[index + 1]
            if flag == "--min-rating":
                options["min_rating"] = self._parse_rating(value)
            elif flag == "--min-votes":
                options["min_votes"] = self._parse_non_negative_int(value)
            elif flag == "--year":
                options["year_from"], options["year_to"] = \
                    self._parse_year_range(value)
            elif flag == "--sort":
                if value not in SORT_OPTIONS:
                    raise ValueError(
                        f"--sort ต้องเป็น {', '.join(SORT_OPTIONS)} "
                        f"(ได้รับ '{value}')"
                    )
                options["sort"] = value
            else:  # --limit
                options["limit"] = self.parse_positive_int(value, "จำนวนรายการ")
            index += 2
        return options

    @staticmethod
    def _parse_rating(text):
        """Convert ``text`` to a 0–10 rating for --min-rating."""
        try:
            value = float(text)
        except ValueError:
            raise ValueError(
                f"ค่า --min-rating ต้องเป็นตัวเลข (ได้รับ '{text}')"
            ) from None
        if not 0.0 <= value <= 10.0:
            raise ValueError("ค่า --min-rating ต้องอยู่ระหว่าง 0 ถึง 10")
        return value

    @staticmethod
    def _parse_non_negative_int(text):
        """Convert ``text`` to a whole number >= 0 for --min-votes."""
        try:
            value = int(text)
        except ValueError:
            raise ValueError(
                f"ค่า --min-votes ต้องเป็นจำนวนเต็ม (ได้รับ '{text}')"
            ) from None
        if value < 0:
            raise ValueError("ค่า --min-votes ต้องไม่ติดลบ")
        return value

    @staticmethod
    def _parse_year_range(text):
        """Convert 'YYYY' or 'YYYY-YYYY' to a (year_from, year_to) pair."""
        parts = text.split("-")
        if len(parts) == 1:
            parts = [parts[0], parts[0]]
        if len(parts) != 2 or not all(
            len(part) == 4 and part.isdigit() for part in parts
        ):
            raise ValueError(
                "ค่า --year ต้องเป็นปี 4 หลัก เช่น 2020 หรือช่วง 1990-2020 "
                f"(ได้รับ '{text}')"
            )
        year_from, year_to = int(parts[0]), int(parts[1])
        if year_from > year_to:
            raise ValueError("ช่วงปีต้องเรียงจากน้อยไปมาก เช่น 1990-2020")
        return year_from, year_to

    # ---------- shared helpers ----------

    @staticmethod
    def describe_tmdb_error(exc):
        """Map a TmdbError to a friendly Thai message."""
        message = str(exc)
        if "401" in message:
            return "คีย์ TMDB ไม่ถูกต้องหรือถูกปฏิเสธ (401) — ตรวจสอบค่า TMDB_API_KEY"
        if "429" in message:
            return "เรียกใช้ TMDB ถี่เกินไป (429) — รอสักครู่แล้วลองใหม่"
        if "404" in message:
            return "ไม่พบข้อมูลที่ขอจาก TMDB (404)"
        return f"เชื่อมต่อ TMDB ไม่สำเร็จ: {message}"

    @staticmethod
    def parse_positive_int(text, label):
        """Convert text to a positive integer or raise a readable error."""
        try:
            value = int(text)
        except ValueError:
            raise ValueError(
                f"{label}ต้องเป็นตัวเลข (ได้รับ '{text}')"
            ) from None
        if value <= 0:
            raise ValueError(f"{label}ต้องมากกว่า 0 (ได้รับ {value})")
        return value

    @staticmethod
    def format_movie(movie):
        """Format one movie dict as a single display line."""
        movie_id = movie.get("id")
        if movie_id is None:
            movie_id = movie.get("tmdb_id")
        year = (movie.get("release_date") or "")[:4] or "----"
        rating = movie.get("vote_average")
        rating_text = "-" if rating is None else f"{rating:.1f}"
        return f"[{movie_id}] {movie['title']} ({year}) คะแนน {rating_text}"


Writing src/cli.py


In [15]:
%%writefile src/app.py
"""Entry point for the interactive Movie Discovery CLI.

Run with:

    python -m src.app
"""

from .cli import CLI


def main():
    """Start the interactive CLI."""
    CLI().run()


if __name__ == "__main__":
    main()


Writing src/app.py


In [16]:
%%writefile tests/conftest.py
"""Pytest configuration: make ``src`` and shared test helpers importable."""

import pathlib
import sys

ROOT = pathlib.Path(__file__).resolve().parents[1]
for path in (ROOT, ROOT / "tests"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))


Writing tests/conftest.py


In [17]:
%%writefile tests/fakes.py
"""Test doubles shared by the test-suite."""

from src.sample_data import SAMPLE_MOVIES
from src.tmdb_client import TmdbError


class FakeTmdbClient:
    """Offline stand-in for TmdbClient with canned responses.

    By default the similar list and the recommendation list overlap on
    movie 103, which exercises the dedupe logic.
    """

    def __init__(self, similar=None, recommendations=None):
        self._similar = list(
            similar if similar is not None else SAMPLE_MOVIES[:3]
        )
        self._recommendations = list(
            recommendations if recommendations is not None else SAMPLE_MOVIES[2:]
        )
        # Call counters so tests can prove when the network was avoided.
        self.search_calls = []
        self.detail_calls = []

    def get_similar(self, tmdb_id, page=1):
        return list(self._similar)

    def get_recommendations(self, tmdb_id, page=1):
        return list(self._recommendations)

    def search_movies(self, query, page=1):
        """Titles containing ``query`` (case-insensitive), like TMDB would."""
        self.search_calls.append(query)
        needle = query.lower()
        return [dict(movie) for movie in SAMPLE_MOVIES
                if needle in movie["title"].lower()]

    def get_movie_details(self, tmdb_id):
        """Details for a sample movie; a 404-style error for unknown ids."""
        self.detail_calls.append(tmdb_id)
        for movie in SAMPLE_MOVIES:
            if movie["id"] == tmdb_id:
                return dict(movie)
        raise TmdbError(
            f"TMDB resource not found (HTTP 404): /movie/{tmdb_id}"
        )


class FakeResponse:
    """Minimal requests.Response stand-in for the client tests."""

    def __init__(self, status_code=200, payload=None):
        self.status_code = status_code
        self._payload = payload

    def json(self):
        if isinstance(self._payload, Exception):
            raise self._payload
        return self._payload


Writing tests/fakes.py


In [18]:
%%writefile tests/test_tmdb_client.py
"""Unit tests for the TMDB client (network calls are mocked)."""

import json
from unittest.mock import Mock

import pytest
import requests

from fakes import FakeResponse
from src.movie_store import MovieStore
from src.tmdb_client import TmdbClient, TmdbError

SEARCH_PAYLOAD = {"results": [{"id": 1, "title": "Inception"},
                              {"id": 2, "title": "Interstellar"}]}


def make_client(session):
    return TmdbClient(api_key="test-key", base_url="https://api.test/3",
                      session=session)


def make_cached_client(session, store, api_key="test-key"):
    return TmdbClient(api_key=api_key, base_url="https://api.test/3",
                      session=session, store=store)


def test_search_movies_returns_results():
    session = Mock()
    session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    client = make_client(session)
    results = client.search_movies("inception")
    assert [movie["id"] for movie in results] == [1, 2]


def test_get_sends_api_key_language_and_query():
    session = Mock()
    session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    client = make_client(session)
    client.search_movies("inception")
    args, kwargs = session.get.call_args
    assert args[0] == "https://api.test/3/search/movie"
    assert kwargs["params"]["api_key"] == "test-key"
    assert kwargs["params"]["language"] == "en-US"
    assert kwargs["params"]["query"] == "inception"


def test_auth_error_raises_tmdb_error():
    session = Mock()
    session.get.return_value = FakeResponse(401)
    client = make_client(session)
    with pytest.raises(TmdbError, match="401"):
        client.search_movies("x")


def test_not_found_raises_tmdb_error():
    session = Mock()
    session.get.return_value = FakeResponse(404)
    client = make_client(session)
    with pytest.raises(TmdbError, match="404"):
        client.get_similar(27205)


def test_rate_limit_raises_tmdb_error():
    session = Mock()
    session.get.return_value = FakeResponse(429)
    client = make_client(session)
    with pytest.raises(TmdbError, match="429"):
        client.get_recommendations(27205)


def test_network_failure_raises_tmdb_error():
    session = Mock()
    session.get.side_effect = requests.ConnectionError("boom")
    client = make_client(session)
    with pytest.raises(TmdbError, match="request to"):
        client.search_movies("x")


def test_invalid_json_raises_tmdb_error():
    session = Mock()
    session.get.return_value = FakeResponse(200, ValueError("bad json"))
    client = make_client(session)
    with pytest.raises(TmdbError, match="invalid JSON"):
        client.search_movies("x")


# ---------- Sprint 2: response cache (store-backed) ----------


def test_cache_hit_skips_the_network():
    store = MovieStore(":memory:")
    session = Mock()
    session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    client = make_cached_client(session, store)
    first = client.search_movies("inception")
    second = client.search_movies("inception")
    assert session.get.call_count == 1
    assert second == first
    assert client.cache_misses == 1
    assert client.cache_hits == 1
    store.close()


def test_cache_key_ignores_the_api_key():
    store = MovieStore(":memory:")
    first_session = Mock()
    first_session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    make_cached_client(first_session, store).search_movies("inception")
    second_session = Mock()
    client = make_cached_client(second_session, store, api_key="other-key")
    results = client.search_movies("inception")
    assert second_session.get.call_count == 0
    assert [movie["id"] for movie in results] == [1, 2]
    store.close()


def test_errors_are_not_cached():
    store = MovieStore(":memory:")
    session = Mock()
    session.get.return_value = FakeResponse(401)
    client = make_cached_client(session, store)
    with pytest.raises(TmdbError, match="401"):
        client.search_movies("x")
    count = store.conn.execute("SELECT COUNT(*) FROM api_cache").fetchone()[0]
    assert count == 0
    store.close()


def test_corrupt_cache_entry_is_refetched():
    store = MovieStore(":memory:")
    session = Mock()
    session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    client = make_cached_client(session, store)
    client.search_movies("inception")
    store.conn.execute("UPDATE api_cache SET payload = 'not json'")
    store.conn.commit()
    results = client.search_movies("inception")
    assert session.get.call_count == 2
    assert [movie["id"] for movie in results] == [1, 2]
    payload = store.conn.execute("SELECT payload FROM api_cache").fetchone()[0]
    assert json.loads(payload)["results"][0]["id"] == 1
    store.close()


def test_expired_cache_entry_is_refetched():
    store = MovieStore(":memory:")
    session = Mock()
    session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    client = make_cached_client(session, store)
    client.search_movies("inception")
    store.conn.execute(
        "UPDATE api_cache SET fetched_at = datetime('now', '-2 days')"
    )
    store.conn.commit()
    client.search_movies("inception")
    assert session.get.call_count == 2
    store.close()


def test_client_without_store_does_not_cache():
    session = Mock()
    session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    client = make_client(session)
    client.search_movies("inception")
    client.search_movies("inception")
    assert session.get.call_count == 2
    assert client.cache_hits == 0
    assert client.cache_misses == 0


def test_get_movie_details_is_cached():
    store = MovieStore(":memory:")
    session = Mock()
    session.get.return_value = FakeResponse(
        200, {"id": 27205, "title": "Inception"}
    )
    client = make_cached_client(session, store)
    first = client.get_movie_details(27205)
    second = client.get_movie_details(27205)
    assert session.get.call_count == 1
    assert first == second == {"id": 27205, "title": "Inception"}
    store.close()


Writing tests/test_tmdb_client.py


In [19]:
%%writefile tests/test_movie_store.py
"""Unit tests for the SQLite persistence layer."""

import pytest

from src.movie_store import CACHE_TTL_SECONDS, MovieStore


@pytest.fixture()
def store(tmp_path):
    db = MovieStore(str(tmp_path / "test.db"))
    yield db
    db.close()


def movie(movie_id=101, title="Alpha Signal", rating=7.5):
    return {"id": movie_id, "title": title, "release_date": "2010-05-01",
            "vote_average": rating, "vote_count": 100, "popularity": 5.0}


def test_add_and_get_movie_roundtrip(store):
    store.add_movie(movie())
    stored = store.get_movie(101)
    assert stored["title"] == "Alpha Signal"
    assert stored["vote_average"] == 7.5


def test_upsert_refreshes_existing_movie(store):
    store.add_movie(movie(rating=7.5))
    store.add_movie(movie(rating=9.1))
    assert len(store.list_movies()) == 1
    assert store.get_movie(101)["vote_average"] == 9.1


def test_missing_movie_returns_none(store):
    assert store.get_movie(999) is None


def test_similar_link_roundtrip(store):
    store.add_similar_link(1, 101, score=8.0)
    store.add_similar_link(1, 102, score=6.0)
    links = store.get_similar_links(1)
    assert [link["target_id"] for link in links] == [101, 102]


def test_watchlist_add_list_and_flag(store):
    store.add_movie(movie())
    assert store.is_in_watchlist(101) is False
    store.add_to_watchlist(101, note="watch soon")
    assert store.is_in_watchlist(101) is True
    rows = store.get_watchlist()
    assert rows[0]["title"] == "Alpha Signal"
    assert rows[0]["note"] == "watch soon"


def test_store_creates_parent_directory(tmp_path):
    nested = tmp_path / "nested" / "dir" / "movies.db"
    db = MovieStore(str(nested))
    db.add_movie(movie())
    db.close()
    assert nested.exists()


# ---------- Sprint 2: watchlist removals, favorites, searches, cache ----------


def test_watchlist_remove_returns_true_then_false(store):
    store.add_movie(movie())
    store.add_to_watchlist(101)
    assert store.remove_from_watchlist(101) is True
    assert store.remove_from_watchlist(101) is False
    assert store.get_watchlist() == []


def test_watchlist_clear_reports_removed_count(store):
    store.add_movie(movie(101))
    store.add_movie(movie(102, title="Bravo Horizon"))
    store.add_to_watchlist(101)
    store.add_to_watchlist(102)
    assert store.clear_watchlist() == 2
    assert store.clear_watchlist() == 0
    assert store.get_watchlist() == []


def test_favorites_roundtrip_and_note_refresh(store):
    store.add_movie(movie())
    assert store.is_in_favorites(101) is False
    store.add_favorite(101, note="top pick")
    assert store.is_in_favorites(101) is True
    rows = store.get_favorites()
    assert rows[0]["title"] == "Alpha Signal"
    assert rows[0]["note"] == "top pick"
    store.add_favorite(101, note="changed")
    assert store.get_favorites()[0]["note"] == "changed"
    assert store.remove_favorite(101) is True
    assert store.remove_favorite(101) is False


def test_recent_searches_newest_first_with_limit(store):
    store.record_search("alpha", 3)
    store.record_search("bravo", 5)
    store.record_search("cobalt", 0)
    rows = store.get_recent_searches(2)
    assert [row["query"] for row in rows] == ["cobalt", "bravo"]
    assert rows[0]["result_count"] == 0
    assert rows[0]["searched_at"]
    assert len(store.get_recent_searches()) == 3


def test_cache_roundtrip_and_expiry(store):
    assert store.get_cached("k") is None
    store.save_cached("k", '{"a": 1}')
    assert store.get_cached("k") == '{"a": 1}'
    store.conn.execute(
        "UPDATE api_cache SET fetched_at = datetime('now', '-2 days') "
        "WHERE cache_key = 'k'"
    )
    store.conn.commit()
    assert store.get_cached("k") is None


def test_cache_save_overwrites_previous_payload(store):
    store.save_cached("k", "old")
    store.save_cached("k", "new")
    assert store.get_cached("k") == "new"


def test_cache_ttl_constant_is_24_hours():
    assert CACHE_TTL_SECONDS == 24 * 60 * 60


Writing tests/test_movie_store.py


In [20]:
%%writefile tests/test_discovery_flow.py
"""End-to-end logic tests: search, discovery filters, builder, export."""

import csv

import pytest

from fakes import FakeTmdbClient
from src.discovery import DiscoveryService
from src.exporter import LIST_COLUMNS, ExportError, Exporter
from src.movie_store import MovieStore
from src.sample_data import SAMPLE_MOVIES
from src.tmdb_client import TmdbError
from src.watchlist import WatchlistBuilder

SEED = 27205


@pytest.fixture()
def store(tmp_path):
    db = MovieStore(str(tmp_path / "s2.db"))
    yield db
    db.close()


def build_service(store, client=None):
    return DiscoveryService(client or FakeTmdbClient(), store)


def test_dedupe_merges_similar_and_recommendations(tmp_path):
    store = MovieStore(str(tmp_path / "a.db"))
    results = build_service(store).find_similar(SEED, limit=10)
    ids = [movie["id"] for movie in results]
    assert len(ids) == len(set(ids))  # movie 103 is in both source lists
    assert 103 in ids
    store.close()


def test_min_rating_filter(tmp_path):
    store = MovieStore(str(tmp_path / "b.db"))
    results = build_service(store).find_similar(SEED, min_rating=8.0)
    assert sorted(movie["id"] for movie in results) == [103, 105]
    store.close()


def test_year_range_excludes_unknown_dates(tmp_path):
    store = MovieStore(str(tmp_path / "c.db"))
    results = build_service(store).find_similar(SEED, year_from=2010,
                                                year_to=2020)
    assert sorted(movie["id"] for movie in results) == [101, 102]  # 104: no date
    store.close()


def test_ranking_prefers_rating_then_votes(tmp_path):
    store = MovieStore(str(tmp_path / "d.db"))
    results = build_service(store).find_similar(SEED)
    assert results[0]["id"] == 103  # 8.4 beats everything else
    store.close()


def test_limit_is_applied(tmp_path):
    store = MovieStore(str(tmp_path / "e.db"))
    results = build_service(store).find_similar(SEED, limit=2)
    assert len(results) == 2
    store.close()


def test_watchlist_builder_skips_saved_movies(tmp_path):
    store = MovieStore(str(tmp_path / "f.db"))
    results = build_service(store).find_similar(SEED)
    builder = WatchlistBuilder(store)
    first = builder.build(results, top_n=2)
    assert len(first) == 2
    again = builder.build(results, top_n=2)
    assert [movie["id"] for movie in again] == \
        [movie["id"] for movie in results[2:4]]
    store.close()


def test_csv_export_roundtrip(tmp_path):
    store = MovieStore(str(tmp_path / "g.db"))
    results = build_service(store).find_similar(SEED)
    WatchlistBuilder(store).build(results, top_n=3)
    out = tmp_path / "watchlist.csv"
    Exporter.export_watchlist(store, str(out))
    with open(out, newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == 3
    assert rows[0]["title"]
    store.close()


# ---------- Sprint 2: search, sorts, add-by-id, favorites, exports ----------


def test_search_caps_at_ten_and_logs_total(store):
    class ManyHits:
        def search_movies(self, query, page=1):
            return [{"id": i, "title": f"T{i}"} for i in range(1, 15)]

    results = DiscoveryService(ManyHits(), store).search("anything")
    assert len(results) == 10
    assert results[0]["title"] == "T1"
    rows = store.get_recent_searches(1)
    assert rows[0]["query"] == "anything"
    assert rows[0]["result_count"] == 14  # total, not just the 10 shown


def test_search_logs_the_query(store):
    client = FakeTmdbClient()
    results = build_service(store, client).search("alpha")
    assert [movie["title"] for movie in results] == ["Alpha Signal"]
    assert client.search_calls == ["alpha"]
    assert store.get_recent_searches(1)[0]["result_count"] == 1


def test_sort_votes_uses_vote_count_first(store):
    ids = [movie["id"] for movie in
           build_service(store).find_similar(SEED, sort="votes")]
    assert ids == [103, 101, 105, 102, 104]


def test_sort_year_puts_unknown_dates_last(store):
    ids = [movie["id"] for movie in
           build_service(store).find_similar(SEED, sort="year")]
    assert ids == [105, 102, 101, 103, 104]


def test_sort_popularity_descending(store):
    ids = [movie["id"] for movie in
           build_service(store).find_similar(SEED, sort="popularity")]
    assert ids == [103, 101, 105, 102, 104]  # 55, 40, 30, 22, 5


def test_unknown_sort_option_raises(store):
    with pytest.raises(ValueError, match="unknown sort"):
        build_service(store).find_similar(SEED, sort="imdb")


def test_only_kept_movies_are_saved_and_linked(store):
    results = build_service(store).find_similar(SEED, min_rating=8.0)
    assert sorted(movie["id"] for movie in results) == [103, 105]
    links = store.get_similar_links(SEED)
    assert [link["target_id"] for link in links] == [103, 105]  # score DESC
    assert store.get_movie(103) is not None
    assert store.get_movie(101) is None  # filtered out -> not stored


def test_ensure_movie_fetches_once_then_uses_store(store):
    client = FakeTmdbClient()
    builder = WatchlistBuilder(store, client)
    first = builder.ensure_movie(101)
    second = builder.ensure_movie(101)
    assert first["title"] == "Alpha Signal"
    assert second["title"] == "Alpha Signal"
    assert client.detail_calls == [101]


def test_add_returns_true_then_false(store):
    client = FakeTmdbClient()
    builder = WatchlistBuilder(store, client)
    assert builder.add(101) is True
    assert builder.add(101) is False
    assert store.is_in_watchlist(101) is True
    assert client.detail_calls == [101]


def test_add_favorite_refreshes_note(store):
    builder = WatchlistBuilder(store, FakeTmdbClient())
    assert builder.add_favorite(102, note="first") is True
    assert builder.add_favorite(102, note="second") is False
    rows = store.get_favorites()
    assert rows[0]["note"] == "second"


def test_ensure_movie_without_client_raises(store):
    builder = WatchlistBuilder(store)
    with pytest.raises(ValueError, match="no TMDB client"):
        builder.ensure_movie(101)


def test_ensure_movie_propagates_404(store):
    builder = WatchlistBuilder(store, FakeTmdbClient())
    with pytest.raises(TmdbError, match="404"):
        builder.ensure_movie(999)


def test_export_favorites_roundtrip(store, tmp_path):
    for entry in SAMPLE_MOVIES[:2]:
        store.add_movie(entry)
        store.add_favorite(entry["id"])
    out = tmp_path / "favorites.csv"
    assert Exporter.export_favorites(store, str(out)) == 2
    with open(out, newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    assert list(rows[0].keys()) == list(LIST_COLUMNS)
    assert {row["title"] for row in rows} == {"Alpha Signal",
                                              "Bravo Horizon"}


def test_export_keeps_tricky_titles_intact(store, tmp_path):
    tricky = {"id": 900, "title": 'Comma, "quoting" & ไทย',
              "release_date": "2024-01-01", "vote_average": 7.0,
              "vote_count": 10, "popularity": 1.0}
    store.add_movie(tricky)
    store.add_to_watchlist(900, note='note, with "quotes"')
    out = tmp_path / "tricky.csv"
    Exporter.export_watchlist(store, str(out))
    with open(out, newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    assert rows[0]["title"] == tricky["title"]
    assert rows[0]["note"] == 'note, with "quotes"'


def test_export_creates_nested_folders(store, tmp_path):
    store.add_movie(dict(SAMPLE_MOVIES[0]))
    store.add_to_watchlist(101)
    out = tmp_path / "deep" / "nested" / "watchlist.csv"
    assert Exporter.export_watchlist(store, str(out)) == 1
    assert out.exists()


def test_export_empty_list_writes_header_only(store, tmp_path):
    out = tmp_path / "empty.csv"
    assert Exporter.export_watchlist(store, str(out)) == 0
    text = out.read_text(encoding="utf-8")
    assert text.strip() == ",".join(LIST_COLUMNS)


def test_export_error_names_the_path(tmp_path):
    store = MovieStore(":memory:")
    blocker = tmp_path / "blocker"
    blocker.write_text("i am a file")
    with pytest.raises(ExportError, match="blocker"):
        Exporter.export_watchlist(store, str(blocker / "out.csv"))
    store.close()


Writing tests/test_discovery_flow.py


In [21]:
%%writefile tests/test_cli.py
"""Unit tests for the interactive CLI.

Inputs are scripted and the services are injected as offline fakes, so
every edge case from the PLAN.md Definition of Done is verified without a
real terminal, network, or database file.
"""

import pytest

from fakes import FakeTmdbClient
from src.cli import CLI
from src.movie_store import MovieStore
from src.sample_data import SAMPLE_MOVIES


def scripted_inputs(lines):
    """Return an input function that replays the given lines, then EOF."""
    queue = list(lines)

    def fake_input(prompt=""):
        if not queue:
            raise EOFError
        return queue.pop(0)

    return fake_input


def run_cli(lines, data_dir=None):
    """Run the CLI with scripted input and fake services; return (outputs, cli)."""
    outputs = []
    cli = CLI(input_func=scripted_inputs(lines), output_func=outputs.append,
              client=FakeTmdbClient(), store=MovieStore(":memory:"),
              data_dir=data_dir)
    cli.run()
    return outputs, cli


def joined(outputs):
    return "\n".join(outputs)


@pytest.mark.parametrize("command", ["quit", "QUIT", "Quit", "  quit  ",
                                     "exit", "q", "ออก"])
def test_quit_variants_exit_with_farewell(command):
    outputs, cli = run_cli([command])
    assert "ลาก่อน" in joined(outputs)
    assert cli.running is False


def test_eof_exits_gracefully():
    outputs, cli = run_cli([])
    assert "ลาก่อน" in joined(outputs)
    assert cli.running is False


def test_empty_input_warns_and_continues():
    outputs, _ = run_cli(["", "quit"])
    assert "กรุณาพิมพ์คำสั่ง" in joined(outputs)
    assert "ลาก่อน" in joined(outputs)


def test_unknown_command_warns_and_recovers():
    outputs, _ = run_cli(["asdf", "quit"])
    text = joined(outputs)
    assert "ไม่รู้จักคำสั่ง" in text
    assert "help" in text
    assert "ลาก่อน" in text


def test_search_requires_a_title():
    outputs, _ = run_cli(["search", "quit"])
    assert "ต้องระบุชื่อเรื่อง" in joined(outputs)


def test_search_finds_sample_movie():
    outputs, _ = run_cli(["search alpha", "quit"])
    assert "Alpha Signal" in joined(outputs)


def test_search_is_case_insensitive():
    outputs, _ = run_cli(["SEARCH ALPHA", "quit"])
    assert "Alpha Signal" in joined(outputs)


def test_discover_rejects_non_numeric_id():
    outputs, _ = run_cli(["discover abc", "quit"])
    assert "ต้องเป็นตัวเลข" in joined(outputs)


def test_discover_rejects_zero_and_negative():
    outputs, _ = run_cli(["discover 0", "discover -3", "quit"])
    assert joined(outputs).count("ต้องมากกว่า 0") == 2


def test_discover_unknown_seed_is_reported():
    outputs, _ = run_cli(["discover 999", "quit"])
    assert "ไม่พบรหัสภาพยนตร์ 999" in joined(outputs)


def test_watchlist_add_list_and_duplicate():
    outputs, cli = run_cli(
        ["watchlist add 101", "watchlist list", "watchlist add 101", "quit"]
    )
    text = joined(outputs)
    assert "เพิ่ม 'Alpha Signal'" in text
    assert "อยู่ในรายการรับชมแล้ว" in text
    assert len(cli.store.get_watchlist()) == 1


def test_watchlist_needs_subcommand():
    outputs, _ = run_cli(["watchlist", "quit"])
    assert "ต้องมีคำสั่งย่อย" in joined(outputs)


def test_help_shows_menu():
    outputs, _ = run_cli(["help", "quit"])
    text = joined(outputs)
    assert "search" in text
    assert "quit" in text


def test_prompt_marker_is_stable():
    prompts = []

    def capture(prompt=""):
        prompts.append(prompt)
        raise EOFError

    cli = CLI(input_func=capture, output_func=lambda *_: None,
              client=FakeTmdbClient(), store=MovieStore(":memory:"))
    cli.run()
    assert prompts and all(prompt == "movie> " for prompt in prompts)


# ---------- Sprint 2: favorites, export, history, flags, missing key ----------


def test_favorites_lifecycle():
    outputs, cli = run_cli(
        ["favorites add 101", "favorites add 101", "favorites list",
         "favorites remove 101", "favorites remove 101", "quit"]
    )
    text = joined(outputs)
    assert "เพิ่ม 'Alpha Signal' เข้ารายการโปรดแล้ว" in text
    assert text.count("อยู่ในรายการโปรดแล้ว") == 1
    assert "รายการโปรด (1 เรื่อง):" in text
    assert "นำ 'Alpha Signal' ออกจากรายการโปรดแล้ว" in text
    assert "ไม่พบรหัสภาพยนตร์ 101 ในรายการโปรด" in text
    assert cli.store.get_favorites() == []


def test_export_writes_csv_to_data_dir(tmp_path):
    outputs, _ = run_cli(
        ["watchlist add 101", "favorites add 102",
         "export watchlist", "export favorites", "quit"],
        data_dir=str(tmp_path)
    )
    text = joined(outputs)
    watch = tmp_path / "watchlist.csv"
    favorites = tmp_path / "favorites.csv"
    assert f"ส่งออก 1 รายการไปที่ {watch}" in text
    assert f"ส่งออก 1 รายการไปที่ {favorites}" in text
    assert "Alpha Signal" in watch.read_text(encoding="utf-8")


def test_export_requires_a_list_name():
    outputs, _ = run_cli(["export wrong", "export", "quit"])
    assert joined(outputs).count("ต้องระบุรายการ") == 2


def test_history_reports_recent_searches():
    outputs, _ = run_cli(["search alpha", "search bravo", "history",
                          "history 1", "quit"])
    text = joined(outputs)
    assert "ประวัติการค้นหาล่าสุด (2 รายการ):" in text
    assert "'alpha'" in text and "'bravo'" in text
    assert text.count("(1 รายการ):") == 1


def test_history_empty_message():
    outputs, _ = run_cli(["history", "quit"])
    assert "ยังไม่มีประวัติการค้นหา" in joined(outputs)


def test_watchlist_remove_and_clear():
    outputs, cli = run_cli(
        ["watchlist add 101", "watchlist remove 101", "watchlist remove 101",
         "watchlist clear", "quit"]
    )
    text = joined(outputs)
    assert "นำ 'Alpha Signal' ออกจากรายการรับชมแล้ว" in text
    assert "ไม่พบรหัสภาพยนตร์ 101 ในรายการรับชม" in text
    assert "ล้างรายการรับชมเรียบร้อยแล้ว (0 เรื่อง)" in text
    assert cli.store.get_watchlist() == []


@pytest.mark.parametrize("command,expected", [
    ("discover 101 --min-rating 11", "ต้องอยู่ระหว่าง 0 ถึง 10"),
    ("discover 101 --min-rating abc", "ต้องเป็นตัวเลข"),
    ("discover 101 --min-votes -5", "ต้องไม่ติดลบ"),
    ("discover 101 --min-votes x", "จำนวนเต็ม"),
    ("discover 101 --year 19x0", "4 หลัก"),
    ("discover 101 --year 2020-1990", "ช่วงปีต้องเรียง"),
    ("discover 101 --sort banana", "--sort ต้องเป็น"),
    ("discover 101 --limit 0", "ต้องมากกว่า 0"),
    ("discover 101 --limit x", "ต้องเป็นตัวเลข"),
    ("discover 101 --bogus 1", "ไม่รู้จักตัวเลือก"),
    ("discover 101 --limit", "ต้องมีค่าตามหลัง"),
])
def test_discover_option_validation(command, expected):
    outputs, _ = run_cli([command, "quit"])
    assert expected in joined(outputs)


def test_discover_sort_and_limit_applied():
    outputs, _ = run_cli(["discover 101 --sort votes --limit 2", "quit"])
    text = joined(outputs)
    assert "เรียงตาม votes" in text
    assert "[103]" in text and "[101]" in text and "[105]" not in text


def test_discover_year_sort_puts_unknown_last():
    outputs, _ = run_cli(["discover 101 --sort year", "quit"])
    text = joined(outputs)
    assert "1. [105]" in text and "5. [104]" in text


def test_missing_key_guides_and_keeps_offline_commands(
        monkeypatch, tmp_path):
    monkeypatch.delenv("TMDB_API_KEY", raising=False)
    outputs = []
    cli = CLI(input_func=scripted_inputs(
                  ["search alpha", "discover 101", "watchlist add 999",
                   "watchlist list", "history", "quit"]),
              output_func=outputs.append, store=MovieStore(":memory:"),
              data_dir=str(tmp_path))
    cli.run()
    text = joined(outputs)
    assert cli.key_missing is True
    assert text.count("ยังไม่ได้ตั้งค่าคีย์ TMDB") == 3
    assert "รายการรับชมยังว่างอยู่" in text
    assert "ยังไม่มีประวัติการค้นหา" in text


def test_offline_add_works_for_stored_movies(monkeypatch):
    monkeypatch.delenv("TMDB_API_KEY", raising=False)
    outputs = []
    store = MovieStore(":memory:")
    store.add_movie(dict(SAMPLE_MOVIES[0]))
    cli = CLI(input_func=scripted_inputs(["watchlist add 101",
                                          "watchlist list", "quit"]),
              output_func=outputs.append, store=store)
    cli.run()
    text = joined(outputs)
    assert "เพิ่ม 'Alpha Signal' เข้ารายการรับชมแล้ว" in text
    assert "[101] Alpha Signal" in text


def test_help_covers_sprint2_commands():
    outputs, _ = run_cli(["help", "quit"])
    text = joined(outputs)
    for command in ("favorites", "export", "history"):
        assert command in text


Writing tests/test_cli.py


### 3.1 สาธิตการทำงานและกรณีขอบเขต (ผลลัพธ์จริงจากการรัน)

เซลล์ถัดไปรันโปรแกรมจริงโดยจำลองอินพุตผู้ใช้ทีละบรรทัด (ไม่ใช่ภาพประกอบ) —
ทุกอย่างออฟไลน์ 100% และผลลัพธ์ที่เห็นคือผลจากการรันจริงในครั้งนี้


In [22]:
# สาธิตการใช้งานจริง (ออฟไลน์ 100%): จำลองผู้ใช้พิมพ์คำสั่งทีละบรรทัด
import pathlib
import tempfile

from fakes import FakeTmdbClient
from src.cli import CLI
from src.movie_store import MovieStore


def scripted(lines):
    """ฟังก์ชันอินพุตจำลอง: คืนค่าทีละบรรทัด แล้วส่ง EOF เมื่ออินพุตหมด"""
    queue = list(lines)

    def feed(prompt=""):
        if not queue:
            raise EOFError
        value = queue.pop(0)
        print(prompt + value)  # แสดงบรรทัดที่ "พิมพ์" เพื่อให้อ่าน transcript ได้
        return value

    return feed


work = pathlib.Path(tempfile.mkdtemp(prefix="sprint2-demo-"))
CLI(input_func=scripted(["search alpha",
                         "discover 101 --min-rating 7 --sort rating --limit 3",
                         "watchlist add 101",
                         "watchlist add 101",
                         "watchlist add 103",
                         "watchlist list",
                         "favorites add 103",
                         "favorites list",
                         "export watchlist",
                         "history",
                         "quit"]),
    output_func=print,
    client=FakeTmdbClient(),
    store=MovieStore(str(work / "movies.db")),
    data_dir=str(work)).run()

print()
print("--- เนื้อหาไฟล์ watchlist.csv ---")
print((work / "watchlist.csv").read_text(encoding="utf-8"))


  Movie Discovery & Watchlist Builder — CLI
  ระบบค้นพบภาพยนตร์และสร้างรายการรับชม
คำสั่งที่ใช้ได้:
  search <ชื่อเรื่อง>              ค้นหาภาพยนตร์จาก TMDB
  discover <รหัส> [ตัวเลือก]       ภาพยนตร์ที่คล้ายกัน (กรอง/เรียงได้)
        --min-rating 0-10 · --min-votes N · --year YYYY|YYYY-YYYY
        --sort rating|votes|year|popularity · --limit N
  watchlist add <รหัส>            เพิ่มเข้ารายการรับชม
  watchlist list                  แสดงรายการรับชม
  watchlist remove <รหัส>         นำออกจากรายการรับชม
  watchlist clear                 ล้างรายการรับชมทั้งหมด
  favorites add <รหัส>            เพิ่มเข้ารายการโปรด
  favorites list                  แสดงรายการโปรด
  favorites remove <รหัส>         นำออกจากรายการโปรด
  export watchlist|favorites      ส่งออกรายการเป็น CSV (โฟลเดอร์ data/)
  history [จำนวน]                 ประวัติการค้นหาล่าสุด (ค่าเริ่มต้น 10)
  help                            แสดงคำสั่งทั้งหมด
  quit / exit / ออก               ออกจากโปรแกรม
movie> search alpha
พบ 1 เรื่อง (

In [23]:
# ทดลองกรณีขอบเขตตาม Definition of Done (ผลลัพธ์จริงจากโปรแกรม)
from fakes import FakeTmdbClient
from src.cli import CLI
from src.movie_store import MovieStore

work2 = pathlib.Path(tempfile.mkdtemp(prefix="sprint2-edge-"))
scenarios = [
    ("1) ค้นหาไม่พบผล", ["search zzz", "quit"]),
    ("2) รหัสภาพยนตร์ไม่มีจริง (404)", ["discover 999", "quit"]),
    ("3) ตัวเลือกผิดรูปแบบ", ["discover 101 --min-rating abc", "quit"]),
    ("4) วิธีเรียงที่ไม่รองรับ", ["discover 101 --sort banana", "quit"]),
    ("5) รหัสหนังผิดรูปแบบ", ["watchlist add abc", "watchlist add -3",
                              "watchlist add 0", "quit"]),
    ("6) อินพุตสิ้นสุด (EOF)", []),
]
for title, lines in scenarios:
    print("=" * 62)
    print(title)
    print("-" * 62)
    CLI(input_func=scripted(lines), output_func=print,
        client=FakeTmdbClient(), store=MovieStore(":memory:"),
        data_dir=str(work2)).run()
    print()


1) ค้นหาไม่พบผล
--------------------------------------------------------------
  Movie Discovery & Watchlist Builder — CLI
  ระบบค้นพบภาพยนตร์และสร้างรายการรับชม
คำสั่งที่ใช้ได้:
  search <ชื่อเรื่อง>              ค้นหาภาพยนตร์จาก TMDB
  discover <รหัส> [ตัวเลือก]       ภาพยนตร์ที่คล้ายกัน (กรอง/เรียงได้)
        --min-rating 0-10 · --min-votes N · --year YYYY|YYYY-YYYY
        --sort rating|votes|year|popularity · --limit N
  watchlist add <รหัส>            เพิ่มเข้ารายการรับชม
  watchlist list                  แสดงรายการรับชม
  watchlist remove <รหัส>         นำออกจากรายการรับชม
  watchlist clear                 ล้างรายการรับชมทั้งหมด
  favorites add <รหัส>            เพิ่มเข้ารายการโปรด
  favorites list                  แสดงรายการโปรด
  favorites remove <รหัส>         นำออกจากรายการโปรด
  export watchlist|favorites      ส่งออกรายการเป็น CSV (โฟลเดอร์ data/)
  history [จำนวน]                 ประวัติการค้นหาล่าสุด (ค่าเริ่มต้น 10)
  help                            แสดงคำสั่งทั้งหมด
  

In [24]:
# โหมดไม่มีคีย์ TMDB: โปรแกรมต้องไม่ล้ม และคำสั่งออฟไลน์ยังใช้ได้
import os

os.environ.pop("TMDB_API_KEY", None)
print("=" * 62)
print("โหมดไม่มีคีย์ TMDB")
print("-" * 62)
CLI(input_func=scripted(["search alpha", "watchlist list", "history", "quit"]),
    output_func=print, client=None, store=MovieStore(":memory:"),
    data_dir=str(work2)).run()


โหมดไม่มีคีย์ TMDB
--------------------------------------------------------------
  Movie Discovery & Watchlist Builder — CLI
  ระบบค้นพบภาพยนตร์และสร้างรายการรับชม
คำสั่งที่ใช้ได้:
  search <ชื่อเรื่อง>              ค้นหาภาพยนตร์จาก TMDB
  discover <รหัส> [ตัวเลือก]       ภาพยนตร์ที่คล้ายกัน (กรอง/เรียงได้)
        --min-rating 0-10 · --min-votes N · --year YYYY|YYYY-YYYY
        --sort rating|votes|year|popularity · --limit N
  watchlist add <รหัส>            เพิ่มเข้ารายการรับชม
  watchlist list                  แสดงรายการรับชม
  watchlist remove <รหัส>         นำออกจากรายการรับชม
  watchlist clear                 ล้างรายการรับชมทั้งหมด
  favorites add <รหัส>            เพิ่มเข้ารายการโปรด
  favorites list                  แสดงรายการโปรด
  favorites remove <รหัส>         นำออกจากรายการโปรด
  export watchlist|favorites      ส่งออกรายการเป็น CSV (โฟลเดอร์ data/)
  history [จำนวน]                 ประวัติการค้นหาล่าสุด (ค่าเริ่มต้น 10)
  help                            แสดงคำสั่งทั้งหมด

In [25]:
# แคชของ TmdbClient: คำตอบเดิมภายใน 24 ชั่วโมงต้องไม่ยิงเครือข่ายซ้ำ
from fakes import FakeResponse
from src.sample_data import SAMPLE_MOVIES
from src.tmdb_client import TmdbClient


class CountingSession:
    """ตัวจำลอง requests.Session: นับจำนวนการยิงเครือข่ายจริง"""

    def __init__(self):
        self.calls = 0

    def get(self, url, params=None, timeout=None):
        self.calls += 1
        return FakeResponse(200, {"results": [SAMPLE_MOVIES[0]]})


work3 = pathlib.Path(tempfile.mkdtemp(prefix="sprint2-cache-"))
store3 = MovieStore(str(work3 / "movies.db"))
session = CountingSession()

client = TmdbClient(api_key="demo-key", session=session, store=store3)
client.search_movies("alpha")
print("เรียกครั้งที่ 1: ยิงเครือข่ายทั้งหมด", session.calls, "ครั้ง |",
      "misses:", client.cache_misses, "hits:", client.cache_hits)

client.search_movies("alpha")
print("เรียกครั้งที่ 2: ยิงเครือข่ายทั้งหมด", session.calls, "ครั้ง |",
      "misses:", client.cache_misses, "hits:", client.cache_hits)

# จำลองว่าผ่านมา 25 ชั่วโมง: ย้อนเวลาแถวแคชในฐานข้อมูล (TTL = 24 ชม.)
with store3.conn:
    store3.conn.execute(
        "UPDATE api_cache SET fetched_at = datetime('now', '-25 hours')")
refreshed = TmdbClient(api_key="demo-key", session=session, store=store3)
refreshed.search_movies("alpha")
print("หลังแคชเก่าเกิน 24 ชม.: ยิงเครือข่ายทั้งหมด", session.calls, "ครั้ง |",
      "misses:", refreshed.cache_misses, "hits:", refreshed.cache_hits)


เรียกครั้งที่ 1: ยิงเครือข่ายทั้งหมด 1 ครั้ง | misses: 1 hits: 0
เรียกครั้งที่ 2: ยิงเครือข่ายทั้งหมด 1 ครั้ง | misses: 1 hits: 1
หลังแคชเก่าเกิน 24 ชม.: ยิงเครือข่ายทั้งหมด 2 ครั้ง | misses: 1 hits: 0


In [26]:
# ข้อมูลอยู่ถาวรข้ามเซสชัน: ปิดโปรแกรมแล้วเปิดใหม่ ข้อมูลต้องอยู่ครบ
work4 = pathlib.Path(tempfile.mkdtemp(prefix="sprint2-persist-"))
db4 = str(work4 / "movies.db")

print("=" * 62)
print("เซสชันที่ 1 — เพิ่มรายการแล้วปิดโปรแกรม")
print("-" * 62)
store_a = MovieStore(db4)
CLI(input_func=scripted(["watchlist add 101", "watchlist add 103", "quit"]),
    output_func=print, client=FakeTmdbClient(), store=store_a,
    data_dir=str(work4)).run()
store_a.close()

print()
print("=" * 62)
print("เซสชันที่ 2 — เปิดโปรแกรมใหม่ ข้อมูลต้องอยู่ครบ")
print("-" * 62)
CLI(input_func=scripted(["watchlist list", "quit"]),
    output_func=print, client=FakeTmdbClient(), store=MovieStore(db4),
    data_dir=str(work4)).run()


เซสชันที่ 1 — เพิ่มรายการแล้วปิดโปรแกรม
--------------------------------------------------------------
  Movie Discovery & Watchlist Builder — CLI
  ระบบค้นพบภาพยนตร์และสร้างรายการรับชม
คำสั่งที่ใช้ได้:
  search <ชื่อเรื่อง>              ค้นหาภาพยนตร์จาก TMDB
  discover <รหัส> [ตัวเลือก]       ภาพยนตร์ที่คล้ายกัน (กรอง/เรียงได้)
        --min-rating 0-10 · --min-votes N · --year YYYY|YYYY-YYYY
        --sort rating|votes|year|popularity · --limit N
  watchlist add <รหัส>            เพิ่มเข้ารายการรับชม
  watchlist list                  แสดงรายการรับชม
  watchlist remove <รหัส>         นำออกจากรายการรับชม
  watchlist clear                 ล้างรายการรับชมทั้งหมด
  favorites add <รหัส>            เพิ่มเข้ารายการโปรด
  favorites list                  แสดงรายการโปรด
  favorites remove <รหัส>         นำออกจากรายการโปรด
  export watchlist|favorites      ส่งออกรายการเป็น CSV (โฟลเดอร์ data/)
  history [จำนวน]                 ประวัติการค้นหาล่าสุด (ค่าเริ่มต้น 10)
  help                        

In [27]:
# รันชุดทดสอบอัตโนมัติทั้ง 93 เคส (จำลองอินพุต/เครือข่าย ไม่ต้องใช้คีย์จริง)
import sys
!{sys.executable} -m pytest -q


........................................................................ [ 77%]
.....................                                                    [100%]
93 passed in 1.23s


In [28]:
# ตรวจสอบมาตรฐานโค้ด (PEP 8)
import sys
!{sys.executable} -m flake8 src tests
print("ผลตรวจ flake8 ด้านบน (ว่าง = ผ่านทั้งหมด)")


ผลตรวจ flake8 ด้านบน (ว่าง = ผ่านทั้งหมด)


In [29]:
# ไฟล์ทั้งหมดที่สร้างขึ้นจากรายงานนี้
import pathlib

skip = {"__pycache__", ".pytest_cache", ".ipynb_checkpoints"}
for path in sorted(pathlib.Path(".").rglob("*")):
    if path.is_dir():
        continue
    if any(part in skip for part in path.parts):
        continue
    print("  " + path.relative_to(".").as_posix())


  .env.example
  requirements.txt
  setup.cfg
  src/__init__.py
  src/app.py
  src/cli.py
  src/config.py
  src/discovery.py
  src/exporter.py
  src/movie_store.py
  src/sample_data.py
  src/tmdb_client.py
  src/watchlist.py
  tests/conftest.py
  tests/fakes.py
  tests/test_cli.py
  tests/test_discovery_flow.py
  tests/test_movie_store.py
  tests/test_tmdb_client.py


## ส่วนที่ 4: ผลลัพธ์และการทดสอบ (Result)

### 4.1 สรุปความก้าวหน้าของงาน (Sprint Progress Summary)

- [x] เชื่อมต่อ TMDB จริงพร้อมแคช 24 ชั่วโมง — พิสูจน์ในเซลล์ "แคชทำงาน": เรียกซ้ำครั้งที่สองไม่ยิงเครือข่ายซ้ำ และเมื่อแคชเก่าเกิน 24 ชั่วโมงระบบจะดึงข้อมูลใหม่ให้อัตโนมัติ
- [x] ฐานข้อมูล SQLite 6 ตารางใช้งานครบ — พิสูจน์ในเซลล์ "ปิด–เปิดโปรแกรมใหม่": รายการยังอยู่ครบ
- [x] คำสั่งชุดเต็มทำงานครบ: `search` · `discover` + ตัวเลือกกรอง/เรียง · `watchlist` · `favorites` · `export` · `history`
- [x] ชุดทดสอบออฟไลน์ 93 เคส ผ่านครบทุกเคส — 42 เคส CLI · 24 เคสการค้นพบ/กรอง/เรียง · 13 เคสคลังข้อมูล · 14 เคสไคลเอนต์ TMDB (ดูผลในเซลล์ pytest ด้านบน)
- [x] flake8 ผ่านทั้งโปรเจกต์ (ผลว่าง = ผ่านทั้งหมด)
- [x] เอกสารประกอบรอบนี้: `sprint2/PLAN.md`
- [ ] ส่งมอบงานผ่าน Pull Request บน GitHub (จะเปิด PR เมื่อถึงขั้นส่งมอบตามข้อตกลงของทีม)

### 4.2 ผลการทดสอบกรณีขอบเขต (QA Report)

| รายการทดสอบ | อินพุต | ผลลัพธ์ที่คาดหวัง | ผลการทดสอบจริง | สถานะ |
|---|---|---|---|---|
| ค้นหาภาพยนตร์ปกติ | `search alpha` | พบเรื่องที่ตรงคำค้น | พบ 1 เรื่อง — [101] Alpha Signal (2010) คะแนน 7.8 | PASSED |
| ค้นพบ + กรอง + เรียง | `discover 101 --min-rating 7 --sort rating --limit 3` | เหลือเฉพาะคะแนน 7 ขึ้นไป เรียงมากไปน้อย | 3 เรื่อง: Cobalt Night 8.4 · Emerald Run 8.0 · Alpha Signal 7.8 | PASSED |
| เพิ่มรายการซ้ำ | `watchlist add 101` สองครั้ง | ครั้งที่สองแจ้งว่ามีอยู่แล้ว | "'Alpha Signal' อยู่ในรายการรับชมแล้ว" | PASSED |
| ข้อมูลอยู่ถาวรข้ามเซสชัน | ปิดและเปิดโปรแกรมใหม่ | รายการเดิมอยู่ครบ | เซสชันที่ 2 แสดง 2 เรื่องเดิม ([101] และ [103]) | PASSED |
| ส่งออกไฟล์ CSV | `export watchlist` | ไฟล์มีหัวตารางและข้อมูลครบ | watchlist.csv — หัวตาราง 5 คอลัมน์ + ข้อมูล 2 แถว | PASSED |
| ประวัติการค้นหา | `history` | แสดงคำค้นล่าสุดพร้อมจำนวนผล | 1 รายการ — 'alpha' พบ 1 ผล | PASSED |
| ค้นหาไม่พบผล | `search zzz` | แจ้งไม่พบโดยไม่ล้ม | "ไม่พบภาพยนตร์ที่ตรงกับ 'zzz'" | PASSED |
| รหัสภาพยนตร์ไม่มีจริง (404) | `discover 999` | แจ้งข้อความอ่านง่าย | "ไม่พบรหัสภาพยนตร์ 999 ใน TMDB" | PASSED |
| ตัวเลือกผิดรูปแบบ | `--min-rating abc` · `--sort banana` | แจ้งเตือนแล้วกลับสู่เมนู | "ค่า --min-rating ต้องเป็นตัวเลข" · "--sort ต้องเป็น rating, votes, ..." | PASSED |
| รหัสหนังผิดรูปแบบ | `watchlist add abc` · `-3` · `0` | ปฏิเสธทุกรูปแบบ | "ต้องเป็นตัวเลข" และ "ต้องมากกว่า 0" ตามลำดับ | PASSED |
| ทำงานโดยไม่มีคีย์ TMDB | เปิดโปรแกรมโดยไม่ตั้ง `TMDB_API_KEY` | แจ้งวิธีตั้งค่า · คำสั่งออฟไลน์ยังใช้ได้ | แสดงข้อความวิธีตั้งค่าคีย์ · watchlist/history ทำงานปกติ | PASSED |
| แคชทำงานและหมดอายุ | เรียกคำเดิมซ้ำ (TTL 24 ชม.) | ซ้ำ = ใช้แคช · เกินอายุ = ดึงใหม่ | ยิงเครือข่าย 1 ครั้ง (hit ในครั้งที่ 2) · หลังย้อนเวลา 25 ชม. ดึงใหม่จริง | PASSED |
| อินพุตสิ้นสุด (EOF) | จบอินพุตทันที | ออกอย่างสุภาพ | "[สิ้นสุดอินพุต]" แล้วกล่าวอำลา | PASSED |

### 4.3 สรุปบทเรียน (Retrospective: Wow! & Whoops!)

**Wow! (ส่วนที่ทำได้ดี):**

- ชั้นข้อมูลและตรรกะธุรกิจครบทั้ง 5 งานตามแผน — TMDB จริง + แคช · SQLite 6 ตาราง · ค้นหา/กรอง/เรียง · CSV · ข้อความข้อผิดพลาดภาษาไทย
- ชุดทดสอบ 93 เคสออฟไลน์ 100% (ไม่ใช้เน็ตและไม่ใช้คีย์) — รันซ้ำได้ทุกเครื่องและบน CI โดยไม่มีความไม่แน่นอนของเครือข่าย
- ข้อมูลคงอยู่ข้ามเซสชันจริง (พิสูจน์ในเซลล์ด้านบน) และพฤติกรรม CLI เดิมจาก Sprint 1 ยังอยู่ครบทุกข้อ

**Whoops! (ปัญหาและแนวทางแก้ไข):**

- เมื่อ CLI เปลี่ยนจากข้อมูลตัวอย่างในตัวเป็นบริการจริง ชุดทดสอบเดิมจาก Sprint 1 (40 เคส) ใช้ต่อไม่ได้ทันที — ทุกเคสต้องถูกปรับให้ "ฉีด" ตัวจำลอง (`FakeTmdbClient`) และฐานข้อมูลชั่วคราว (`:memory:`) ผ่านพารามิเตอร์ของ CLI แทน งานรอบนี้จึงรวมการรื้อชุดทดสอบยกชุด ผลลัพธ์คือรูปแบบ injection กลายเป็นมาตรฐานของทั้งโครงการ
- หนี้จาก Sprint 1: รายการรับชมเคยเก็บในหน่วยความจำชั่วคราว ข้อมูลหายเมื่อปิดโปรแกรม — รอบนี้ปิดได้แล้วด้วยฐานข้อมูล SQLite ที่เก็บถาวร

### 4.4 ลิงก์และขั้นตอนถัดไป

- **Repository:** https://github.com/somjung/Movie-Discovery
- **เอกสารประกอบ:** `sprint2/PLAN.md` (แผนงาน)
- **Sprint ถัดไป (Sprint 3 — Full-Stack):** เชื่อมชั้นนำเสนอรูปแบบใหม่ (เว็บแอปพลิเคชัน) เข้ากับบริการชุดเดิมของรอบนี้ โดยไม่แก้โค้ด `src/` ที่ส่งมอบแล้ว


---

*หมายเหตุ: เซลล์สาธิตทั้งหมดในรายงานนี้รันแบบออฟไลน์ 100% — ใช้ตัวจำลอง TMDB ชุดเดียวกับชุดทดสอบ (`tests/fakes.py`) และฐานข้อมูลชั่วคราว จึงไม่ต้องใช้คีย์หรืออินเทอร์เน็ต และรันซ้ำได้ทุกเครื่อง · โน้ตบุ๊กนี้สร้างจากไฟล์โค้ดจริงในโปรเจกต์ด้วยสคริปต์ `build_notebook.py` (อยู่ในโฟลเดอร์เดียวกับโน้ตบุ๊ก) · This product uses the TMDB API but is not endorsed or certified by TMDB.*
